# Delta Norm Ratio vs Accuracy Analysis (cleaned)

Trimmed copy of `delta_norm_epoch_analysis.ipynb` for sharing: keeps only the analyses that
made it into the report --

- **Task 3**: Spearman rho heatmap (masked by significance), per group.
- **Task 4**: norm ratio through training, per subset.
- **Task 5a**: layer-wise final accuracy per subset (Task 5 + Task 6's plot styling).
- **Task 7**: layer-wise accuracy over epochs.

Everything specific to the dropped analyses (Task 1/2/2b's scatter plots, Task 5's plain
styling, Task 6, Task 8, and the R^2/p-value plot) has been removed, along with the
data configs and dataframes only those used. Local absolute paths are replaced with a
`DATA_ROOT`/`OUTPUT_DIR` you edit once below.

This still relies on the same "Groups" data-grouping system as the original (Groups 1-8,
`json_maps_by_subset`) -- Tasks 3/4/5a/7 all look up a subset by name from that dict, so the
group-building cells stay in full, comments and all (the notebook's own docs mark those
comments as an intentional audit trail of excluded rows, not leftover clutter).

In [ ]:
import matplotlib.pyplot as plt
import os
import glob
import json
import pandas as pd
import numpy as np
import seaborn as sns
from tueplots import bundles
plt.rcParams.update(bundles.iclr2024())

NO_SPINES = True  # set False to restore default matplotlib top/right spines on all plots in this notebook

if NO_SPINES:
    plt.rcParams.update({
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

# Edit these for your machine before running.
DATA_ROOT = "/path/to/metrics"
OUTPUT_DIR = "/path/to/output/scaling_hypothesis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if NO_SPINES:
    OUTPUT_DIR_PNG = "/path/to/output/pngs"
    OUTPUT_DIR_PDF = "/path/to/output/pdfs"
else:
    OUTPUT_DIR_PNG = OUTPUT_DIR
    OUTPUT_DIR_PDF = OUTPUT_DIR
os.makedirs(OUTPUT_DIR_PNG, exist_ok=True)
os.makedirs(OUTPUT_DIR_PDF, exist_ok=True)

PARENT_PATH_IMNET100_SMALL = os.path.join(DATA_ROOT, "imnet100_small")
PARENT_PATH_IMNET_BASE = os.path.join(DATA_ROOT, "imnet_base")


## Helper functions

Copied from `report_plots.ipynb` (`# utils` and `# process (Acc)` sections), plus
`get_attribute_training_with_epochs`, which returns the epoch values alongside the metric
values so epoch-49 lookups and epoch-bucket stats can match on the *actual* epoch value
rather than assuming a fixed position in an epoch list.

In [ ]:
def load_json_with_continuations(full_path):
    """Load a metrics json file, merging any '_s0'/'_s1'/'_s2' continuation files next to it.

    Logging for a run sometimes got cut off partway through training and the rest of the
    epochs were logged into a separate per-seed file named like '<base>_s0.json',
    '<base>_s1.json', '<base>_s2.json' (one continuation file per seed) instead of being
    appended to the original file. Each continuation file has the same top-level structure
    as the base file but its 'ft' list holds only the one seed's remaining stats entries.
    This merges those extra stats entries into the matching seed so no epochs are silently
    dropped, and prints what it did (or didn't find) so gaps are traceable to a specific file.
    """
    with open(full_path) as f:
        data = json.load(f)

    base_no_ext, ext = os.path.splitext(full_path)
    continuation_paths = sorted(glob.glob(f"{base_no_ext}_s[0-9]*{ext}"))
    if not continuation_paths:
        return data

    print(f"Found {len(continuation_paths)} continuation file(s) for {full_path}: {[os.path.basename(p) for p in continuation_paths]}")

    for cont_path in continuation_paths:
        try:
            with open(cont_path) as f:
                cont_data = json.load(f)
        except Exception as e:
            print(f"*** ERROR ***: Failed to load continuation file {cont_path} for base {full_path}: {e}")
            continue

        if len(cont_data) != len(data):
            print(f"*** WARNING ***: Continuation file {cont_path} has {len(cont_data)} run(s) but base file {full_path} has {len(data)}; merging only overlapping runs by position.")

        for run_idx, cont_run in enumerate(cont_data):
            if run_idx >= len(data):
                print(f"*** WARNING ***: Continuation file {cont_path} has an extra run at index {run_idx} with no matching run in base file {full_path}; skipping it.")
                continue
            base_run = data[run_idx]
            base_ft_by_seed = {ft_entry.get("seed"): ft_entry for ft_entry in base_run.get("ft", [])}
            for cont_ft in cont_run.get("ft", []):
                seed = cont_ft.get("seed")
                n_new = len(cont_ft.get("stats", []))
                if seed in base_ft_by_seed:
                    base_ft_by_seed[seed].setdefault("stats", []).extend(cont_ft.get("stats", []))
                    print(f"Merged {n_new} continuation stats entries from {os.path.basename(cont_path)} into seed {seed} (run {run_idx}) of {os.path.basename(full_path)}")
                else:
                    print(f"*** WARNING ***: Seed {seed} from {os.path.basename(cont_path)} not found among base seeds {list(base_ft_by_seed.keys())} in {os.path.basename(full_path)}; appending as a new 'ft' entry instead of merging.")
                    base_run.setdefault("ft", []).append(cont_ft)
                    base_ft_by_seed[seed] = cont_ft
    return data


def lists_extend(lists):
    master_list = []
    for lst in lists:
        master_list.extend(lst)
    return master_list

def element_wise_mean(lists):
    max_length = max(len(lst) for lst in lists)
    padded_lists = [lst + [float('nan')] * (max_length - len(lst)) for lst in lists]
    df = pd.DataFrame(padded_lists)
    return df.mean(skipna=True).tolist()

def element_wise_std(lists):
    max_length = max(len(lst) for lst in lists)
    padded_lists = [lst + [float('nan')] * (max_length - len(lst)) for lst in lists]
    df = pd.DataFrame(padded_lists)
    return df.std(skipna=True).tolist()

def get_layer_accuracies(row):
    label = row["label"]
    tag = row.get("tag", "No tag")
    path = row.get("path", "Unknown path")
    seed = row.get("seed", "Unknown seed")
    df_stats_all = pd.DataFrame(row["stats"])
    if "epoch" not in df_stats_all.columns or "layer_sub" not in df_stats_all.columns:
        print(f"*** ERROR ***: Missing 'epoch' or 'layer_sub' column for label={label!r}, tag={tag!r}, seed={seed!r}, path={path!r}. Columns found: {df_stats_all.columns.tolist()}")
    df_stats = df_stats_all[(~df_stats_all["layer_sub"].isna()) & (df_stats_all["epoch"] == 299)].head(24)
    if df_stats.empty:
        available_epochs = df_stats_all["epoch"].unique().tolist() if "epoch" in df_stats_all.columns else "N/A"
        print(f"*** ERROR ***: No rows with epoch==299 for label={label!r}, tag={tag!r}, seed={seed!r}, path={path!r}. Available epochs: {available_epochs}")
    if "acc" in df_stats.columns:
        return_list = df_stats["acc"].to_list()
    else:
        if "attn_acc" not in df_stats.columns or "blk_acc" not in df_stats.columns:
            print(f"*** ERROR ***: No 'acc', 'attn_acc', or 'blk_acc' column for label={label!r}, tag={tag!r}, seed={seed!r}, path={path!r}. Columns found: {df_stats.columns.tolist()}")
        df_stats["acc"] = df_stats.apply(
            lambda x: x["attn_acc"] if not pd.isna(x["attn_acc"]) else x["blk_acc"], axis=1
        )
        return_list = df_stats["acc"].to_list()
    if len(return_list) != 24:
        print(f"Warning: Expected 24 accuracies, but got {len(return_list)} for label={label!r}, tag={tag!r}, seed={seed!r}, path={path!r}.")
        if len(return_list) == 48:
            return_list = return_list[::2]
    if len(return_list) == 0:
        print(f"*** ERROR ***: layer_accuracies is EMPTY for label={label!r}, tag={tag!r}, seed={seed!r}, path={path!r}. Downstream 'final_accuracies' for this row will be NaN.")
    return return_list

def get_attribute_training_with_epochs(row, column_name, layer_sub):
    """Like get_attribute_training, but also returns the epoch each value belongs to,
    so downstream code can match on the real epoch instead of assuming list position."""
    df_stats = pd.DataFrame(row["stats"])
    df_stats = df_stats.dropna(subset=[column_name], ignore_index=True).dropna(subset=["epoch"], ignore_index=True)
    df_stats = df_stats[df_stats["layer_sub"] == layer_sub].drop_duplicates(subset=["epoch"], keep="first").sort_values("epoch")
    return df_stats["epoch"].to_list(), df_stats[column_name].to_list()


## Build `dft` (per-model, per-seed-averaged) dataframe

Same as `process_jsons_training` in `report_plots.ipynb`, extended to also capture the
epochs for every `delta_norm_ratio_{layer}{attn|mlp}` series (`..._epochs` columns). Used by
the Groups-building cells below and by Task 4's driver; each caller sets the module-global
`parent_path` (or passes it explicitly, for the newer helpers) right before calling this.

In [ ]:
def process_jsons_training(json_map):
    all_data = []
    for json_item in json_map:
        json_path = json_item["path"]
        print(f"Processing {json_path}...")
        file_path = os.path.join(parent_path, json_path)
        if not os.path.exists(file_path):
            print(f"*** ERROR ***: File not found: {file_path} (from json_map entry: {json_item})")
            continue
        if os.path.getsize(file_path) == 0:
            print(f"*** ERROR ***: {file_path} is empty on disk, skipping.")
            continue
        try:
            data = load_json_with_continuations(file_path)
        except Exception as e:
            print(f"*** ERROR ***: Failed to load/parse {file_path} (from json_map entry: {json_item}): {e}")
            continue
        if not data:
            print(f"*** ERROR ***: {file_path} loaded but is empty (from json_map entry: {json_item})")
        for item in data:
            item["category"] = json_item.get("category", "")
            item["tag"] = json_item.get("tag", "")
            item["init"] = json_item.get("init", "")
            item["layer"] = json_item.get("layer", "")
            item["path"] = json_path
        all_data.extend(data)
    if not all_data:
        raise ValueError(f"*** ERROR ***: No data was loaded for any file in json_map: {[j.get('path') for j in json_map]}")
    df = pd.DataFrame(all_data)

    df = df.explode("ft").reset_index(drop=True).dropna(subset=["ft"], ignore_index=True)
    df["stats"] = df["ft"].apply(lambda x: x["stats"])
    df["label"] = df.apply(lambda x: f"{x['init']}: {x['category']}: {x['layer']}: {x['tag']}", axis=1)
    df["seed"] = df["ft"].apply(lambda x: x["seed"])

    df_columns = df.columns.tolist()
    df = df.groupby(["label", "seed"]).agg({
        "stats": lists_extend,
        **{col: "first" for col in df_columns if col not in ["stats", "label", "seed"]}
    }).reset_index()

    df["layer_accuracies"] = df.apply(lambda x: get_layer_accuracies(x), axis=1)

    for ix in range(0, 12):
        for kind, layer_sub in [("attn", ix - 0.5), ("mlp", ix)]:
            col = f"delta_norm_ratio_{ix}{kind}"
            epochs_and_values = df.apply(
                lambda x: get_attribute_training_with_epochs(x, "delta_norm_ratio", layer_sub), axis=1
            )
            df[f"{col}_epochs"] = epochs_and_values.apply(lambda x: x[0])
            df[col] = epochs_and_values.apply(lambda x: x[1])

    value_cols = [f"delta_norm_ratio_{ix}{kind}" for ix in range(12) for kind in ("attn", "mlp")]
    epoch_cols = [f"{c}_epochs" for c in value_cols]

    agg_dict = {
        "layer_accuracies": element_wise_mean,
        "category": "first",
        "init": "first",
        "tag": "first",
        "layer": "first",
        "path": "first",
        "seed": "count",
    }
    df_grouped = df.groupby(["label"]).agg(agg_dict).reset_index()
    df_grouped["layer_accuracies_std"] = df.groupby(["label"]).agg(
        {"layer_accuracies": element_wise_std}
    ).reset_index()["layer_accuracies"]

    df_grouped["final_accuracies"] = df_grouped["layer_accuracies"].apply(lambda x: x[-1] if len(x) > 0 else float("nan"))
    df_grouped["final_accuracies_std"] = df_grouped["layer_accuracies_std"].apply(lambda x: x[-1] if len(x) > 0 else float("nan"))

    # Random-init baseline checkpoint (s6257433.json) is hardcoded to its known accuracy.
    df_grouped.loc[df_grouped["path"] == "s6257433.json", "final_accuracies"] = 84.37

    for col in value_cols:
        df_grouped[col] = df.groupby(["label"]).agg({col: element_wise_mean}).reset_index()[col]
        df_grouped[f"{col}_std"] = df.groupby(["label"]).agg({col: element_wise_std}).reset_index()[col]

    # epochs should line up across seeds for the same label; just take the first seed's epoch list
    for ecol in epoch_cols:
        df_grouped[ecol] = df.groupby(["label"]).agg({ecol: "first"}).reset_index()[ecol]

    if any(df_grouped["seed"] != 3):
        print("Warning: Some groups do not have 3 seeds. Here are those rows:")
        print(df_grouped[df_grouped["seed"] != 3])
    return df_grouped


## Shared plotting helpers

`_is_vitb_leaf` and the two `PARENT_PATH_*` constants route a subset name to the right data
directory (ViT-Base subsets live under a different `parent_path` than ViT-Small ones);
`save_fig` saves a figure as PNG+PDF into `OUTPUT_DIR_PNG`/`OUTPUT_DIR_PDF`. Both are used by
every Task below.

In [ ]:
import matplotlib.lines as mlines
from scipy.stats import spearmanr


def _is_vitb_leaf(name):
    """ViT-Base leaves/subsets are either the numbered Group-4 leaves ("4", "4A_early", ...)
    or the hand-provided VITB_BASELINE subset -- both live under PARENT_PATH_IMNET_BASE,
    everything else under PARENT_PATH_IMNET100_SMALL."""
    return name.startswith("4") or name.startswith("VITB")


def save_fig(fig, path_without_ext, formats=("png", "pdf")):
    """Save a figure in multiple formats, routing each format to its spine-conditional dir
    (OUTPUT_DIR_PNG / OUTPUT_DIR_PDF) regardless of the directory baked into path_without_ext."""
    basename = os.path.basename(path_without_ext)
    format_dirs = {"png": OUTPUT_DIR_PNG, "pdf": OUTPUT_DIR_PDF}
    paths = {}
    for fmt in formats:
        path = os.path.join(format_dirs[fmt], f"{basename}.{fmt}")
        fig.savefig(path, dpi=1500 if fmt == "png" else None)
        paths[fmt] = path
    return paths


# Experimental Data Groupings for R-Squared Analysis

**Pre-processing Requirement:** Before calculating $R^2$ or slopes for any of these groups, you must filter out "crashed" runs (e.g., runs that flatlined at random chance accuracy). Crashes represent a discontinuous failure in gradient flow and will artificially destroy the linear regression fit for the surviving models.

---

## Group 1: The Unified Random Baseline (Scaling Ablation)
**Configuration:** Random-init weights paired with PR-init Norm Ratios.
**Statistical Rationale:** Because the underlying structural capacity is identical across all these runs (standard random initialization), Simpson's Paradox does not apply. You can safely combine the available PR norm sources into one regression model to increase statistical power. 
*(Note: Scaling ablations were only run on D4 and Shuffled-D98).*

*   **Subset 1A:** k-Dyck D4 + k-Dyck Shuffled-D98 — **Early-to-Late sweep**
*   **Subset 1B:** k-Dyck D4 + k-Dyck Shuffled-D98 — **Late-to-Early sweep**

---

## Group 2: The Pure PR Runs
**Configuration:** PR-init weights paired with PR-init Norm Ratios.
**Statistical Rationale:** The structural knowledge (inductive bias) of the weights changes depending on the synthetic dataset. You must split these by dataset source to account for different baseline accuracies (Y-intercepts) and optimal gradient flow ratios (X-axis shifts). This is available for all 4 variants.

*   **Subset 2A:** k-Dyck D4 Only (Early-to-Late vs. Late-to-Early)
*   **Subset 2B:** k-Dyck Shuffled-D98 Only (Early-to-Late vs. Late-to-Early)
*   **Subset 2C:** k-Dyck Truncated-D98 Only (Early-to-Late vs. Late-to-Early)
*   **Subset 2D:** k-Dyck Truncated-Shuffled-D4 Only (Early-to-Late vs. Late-to-Early)

---

## Group 3: PR Weights with Random Ratios (Scaling Ablation)
**Configuration:** PR-init weights paired with Random-init Norm Ratios.
**Statistical Rationale:** Follows the same logic as Group 2. The varying weight sources dictate the capacity, meaning they must remain separated by pretraining source. 
*(Note: Scaling ablations were only run on D4 and Shuffled-D98).*

*   **Subset 3A:** k-Dyck D4 Only (Early-to-Late vs. Late-to-Early)
*   **Subset 3B:** k-Dyck Shuffled-D98 Only (Early-to-Late vs. Late-to-Early)

---

## Group 4: The Architecture Check (ViT-Base)
**Configuration:** Runs from `json_map_vitb_kd4` (ViT-Base Architecture).
**Statistical Rationale:** Variance accumulation scales proportionally with the square root of layer depth ($1/\sqrt{l}$). ViT-Base has a fundamentally different depth and width than ViT-Small, meaning its mathematical baseline for a "healthy" norm ratio is physically different. It must never be mixed with ViT-Small data.

*   **Subset 4A:** ViT-Base Pure PR (Early-to-Late vs. Late-to-Early)
*   **Subset 4B:** ViT-Base PR Weights / Random Ratios (Early-to-Late vs. Late-to-Early)
*   **Subset 4C:** ViT-Base Random Weights / PR Ratios (Early-to-Late vs. Late-to-Early)

## Building the R²-analysis subsets (splits only)

Builds one literal `json_map_group*` list per leaf subset named in the groupings above (20 leaves: `1A`/`1B`, and `2A`-`4C` each split into `_early`/`_late`), plus a `json_maps_by_subset` dict that collects them (flat list for `1A`/`1B`; `{"early": ..., "late": ...}` dict for the rest, since those subsets are compared across both directions). This step only performs the splitting/filtering -- processing into dataframes, crash-filtering, R²/slope fitting, and plotting are left for the follow-up task.

**Sourcing (mirrors the master `json_map_scaling` / `json_map_rr_pr_pp_rp` / `json_map_vitb_kd4` cells exactly, including their commented-out rows):**
- **Group 1** (`1A`, `1B`): `Rand-init w/ PR-init Norm Ratios` category from `json_map_scaling`, split by direction, with `k-Dyck - D4` and `k-Dyck Shuffled - D98` entries combined (per the group's statistical rationale).
- **Group 2** (`2A`-`2D`): pulled entirely from `json_map_rr_pr_pp_rp` -- `Procedural-init / Random-init` (part before `/`) for early-to-late, `Random-init / Procedural-init` (part after `/`) for late-to-early -- filtered per dataset, since this is the only source covering all 4 k-Dyck variants. **`2E`-`2H`** are later additions: one single L0-11 Pure-PR checkpoint each for the 4 k-Dyck variants that don't have a partial layer-scaling sweep (D98, Truncated-D4, Shuffled-D4, Truncated-Shuffled-D98) -- see the Groups 5/6 cell below for how their early/late leaves are populated.
- **Group 3** (`3A`, `3B`): `PR-init w/ Rand-init Norm Ratios` category from `json_map_scaling`, split by direction and by dataset (`k-Dyck - D4` only, `k-Dyck Shuffled - D98` only).
- **Group 4** (`4A`-`4C`): from `json_map_vitb_kd4`, same three category patterns as Groups 1-3 (Pure PR via the `/`-slash categories, `PR-init w/ Rand-init Norm Ratios`, `Rand-init w/ PR-init Norm Ratios`), split by direction.

Direction (`early`/`late`) is classified from the `layer` string: early-to-late layers match `L0` or `L0-N` (growing forward from layer 0), late-to-early layers match `L11` or `LN-11` (growing backward from layer 11). **`L0-11` (the fully-scaled endpoint, all 12 layers on one side) matches both patterns and is deliberately included in *both* the `_early` and `_late` leaf for its group** -- it's the shared limit both sweep directions converge on, not an early- or late-only point, so dropping it from either (or both) would discard real data. For Group 4A specifically, `json_map_vitb_kd4`'s slash-based categories can't represent it (both parts of `X / Y` must be non-empty), so it's logged there under a separate plain `PR` category instead and folded into `4A_early`/`4A_late` the same way.

**Rows that are commented out in the master maps (`json_map_scaling`, `json_map_rr_pr_pp_rp`, `json_map_vitb_kd4`) stay commented out here too** -- they're listed for traceability but not included as active data, matching the master lists' current intentional exclusions (e.g. `1A` has 0 active rows: no early-direction random-baseline runs are currently active for D4/ksd98 in the source, only commented ones). `4B_early` and `4C_early` are non-empty only because of the shared `L0-11` endpoint above -- the ViT-Base early-direction PR/random-mismatch runs are otherwise absent from the source.


In [ ]:
# --- Group 1A: Unified Random Baseline (Rand-init weights + PR-init norm ratios), D4+ksd98 combined, early-to-late ---
json_map_group1a = [
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-8',
    #     'path': 's6742188.json',
    #     'init': 'k-Dyck - D4',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-9',
    #     'path': 's6742190.json',
    #     'init': 'k-Dyck - D4',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-10',
    #     'path': 's6742191.json',
    #     'init': 'k-Dyck - D4',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-11',
    #     'path': 's6098383.json',
    #     'init': 'k-Dyck - D4',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0',
    #     'path': 's6742006.json',
    #     'init': 'k-Dyck Shuffled - D98',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-1',
    #     'path': 's6742008.json',
    #     'init': 'k-Dyck Shuffled - D98',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-2',
    #     'path': 's6742009.json',
    #     'init': 'k-Dyck Shuffled - D98',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-3',
    #     'path': 's6742010.json',
    #     'init': 'k-Dyck Shuffled - D98',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-4',
    #     'path': 's6742011.json',
    #     'init': 'k-Dyck Shuffled - D98',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-5',
    #     'path': 's6742012.json',
    #     'init': 'k-Dyck Shuffled - D98',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-6',
    #     'path': 's6742013.json',
    #     'init': 'k-Dyck Shuffled - D98',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-7',
    #     'path': 's6742014.json',
    #     'init': 'k-Dyck Shuffled - D98',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-8',
    #     'path': 's6742015.json',
    #     'init': 'k-Dyck Shuffled - D98',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-9',
    #     'path': 's6742016.json',
    #     'init': 'k-Dyck Shuffled - D98',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-10',
    #     'path': 's6742017.json',
    #     'init': 'k-Dyck Shuffled - D98',
    # },
    # {
    #     'category': 'Rand-init w/ PR-init Norm Ratios',
    #     'layer': 'L0-11',
    #     'path': 's6707091.json',
    #     'init': 'k-Dyck Shuffled - D98',
    # },
    {
        'category': '',
        'layer': 'L0-11',
        'path': 's6257433.json',
        'init': 'Random-init',
    },
]

# --- Group 1B: Unified Random Baseline, D4+ksd98 combined, late-to-early ---
json_map_group1b = [
    {
        'category': '',
        'layer': 'L0-11',
        'path': 's6257433.json',
        'init': 'Random-init',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L0-11',
        'path': 's6098383.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L1-11',
        'path': 's6098384.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L2-11',
        'path': 's6098385.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L3-11',
        'path': 's6098386.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L4-11',
        'path': 's6098387.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L5-11',
        'path': 's6098388.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L6-11',
        'path': 's6098391.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L7-11',
        'path': 's6098396.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L8-11',
        'path': 's6098397.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L9-11',
        'path': 's6098398.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L10-11',
        'path': 's6098399.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L11',
        'path': 's6098401.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L0-11',
        'path': 's6707091.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L1-11',
        'path': 's6707052.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L2-11',
        'path': 's6707053.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L3-11',
        'path': 's6707054.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L4-11',
        'path': 's6707055.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L5-11',
        'path': 's6707058.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L6-11',
        'path': 's6162896.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L7-11',
        'path': 's6162894.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L8-11',
        'path': 's6162892.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L9-11',
        'path': 's6162890.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L10-11',
        'path': 's6162888.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L11',
        'path': 's6707089.json',
        'init': 'k-Dyck Shuffled - D98',
    },
]

# --- Group 2A: Pure PR, k-Dyck - D4, early-to-late ---
json_map_group2a_early = [
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0',
        'path': 's6320110.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-1',
        'path': 's6322567.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-2',
        'path': 's6316128.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-3',
        'path': 's6316129.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-4',
        'path': 's6316130.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-5',
        'path': 's6316131.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-6',
        'path': 's6316132.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-7',
        'path': 's6316137.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-8',
        'path': 's6316138.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-9',
        'path': 's6316139.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-10',
        'path': 's6316140.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-11',
        'path': 's6848384.json',
        'init': 'k-Dyck - D4',
    }
]

# --- Group 2A: Pure PR, k-Dyck - D4, late-to-early ---
json_map_group2a_late = [
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L0-11',
        'path': 's6848384.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L1-11',
        'path': 's6089438.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L2-11',
        'path': 's6089440.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L3-11',
        'path': 's6089439.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L4-11',
        'path': 's6089441.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L5-11',
        'path': 's6089442.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L6-11',
        'path': 's6089443.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L7-11',
        'path': 's6089444.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L8-11',
        'path': 's6089445.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L9-11',
        'path': 's6089446.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L10-11',
        'path': 's6089447.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L11',
        'path': 's6316141.json',
        'init': 'k-Dyck - D4',
    },
]

# --- Group 2B: Pure PR, k-Dyck Shuffled - D98, early-to-late ---
json_map_group2b_early = [
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0',
        'path': 's6329850.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-1',
        'path': 's6329849.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-2',
        'path': 's6329848.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-3',
        'path': 's6329847.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-4',
        'path': 's6329844.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-5',
        'path': 's6329843.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-6',
        'path': 's6329842.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-7',
        'path': 's6339953.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-8',
        'path': 's6329840.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-9',
        'path': 's6329839.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-10',
        'path': 's6329838.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-11',
        'path': 's6848396.json',
        'init': 'k-Dyck Shuffled - D98',
    },
]

# --- Group 2B: Pure PR, k-Dyck Shuffled - D98, late-to-early ---
json_map_group2b_late = [
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L0-11',
        'path': 's6848396.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L1-11',
        'path': 's6340034.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L2-11',
        'path': 's6340035.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L3-11',
        'path': 's6340036.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L4-11',
        'path': 's6340037.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L5-11',
        'path': 's6340038.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L6-11',
        'path': 's6340039.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L7-11',
        'path': 's6340040.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L8-11',
        'path': 's6340042.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L9-11',
        'path': 's6340043.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L10-11',
        'path': 's6340044.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L11',
        'path': 's6340045.json',
        'init': 'k-Dyck Shuffled - D98',
    },
]

# --- Group 2C: Pure PR, k-Dyck Truncated - D98, early-to-late ---
json_map_group2c_early = [
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0',
        'path': 's6379049.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-1',
        'path': 's6379050.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-2',
        'path': 's6379051.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-3',
        'path': 's6379052.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-4',
        'path': 's6379053.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-5',
        'path': 's6379054.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-6',
        'path': 's6379056.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-7',
        'path': 's6379057.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-8',
        'path': 's6379252.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-9',
        'path': 's6379257.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-10',
        'path': 's6379258.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-11',
        'path': 's6848392.json',
        'init': 'k-Dyck Truncated - D98',
    },
]

# --- Group 2C: Pure PR, k-Dyck Truncated - D98, late-to-early ---
json_map_group2c_late = [
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L0-11',
        'path': 's6848392.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L1-11',
        'path': 's6364580.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L2-11',
        'path': 's6366195.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L3-11',
        'path': 's6366200.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L4-11',
        'path': 's6366201.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L5-11',
        'path': 's6366205.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L6-11',
        'path': 's6366206.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L7-11',
        'path': 's6366207.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L8-11',
        'path': 's6366208.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L9-11',
        'path': 's6366210.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L10-11',
        'path': 's6366211.json',
        'init': 'k-Dyck Truncated - D98',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L11',
        'path': 's6366212.json',
        'init': 'k-Dyck Truncated - D98',
    },
]

# --- Group 2D: Pure PR, k-Dyck Truncated Shuffled - D4, early-to-late ---
json_map_group2d_early = [
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0',
        'path': 's6445996.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-1',
        'path': 's6445997.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-2',
        'path': 's6446000.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-3',
        'path': 's6446001.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-4',
        'path': 's6446002.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-5',
        'path': 's6446003.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-6',
        'path': 's6446004.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-7',
        'path': 's6446005.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-8',
        'path': 's6446006.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-9',
        'path': 's6446007.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-10',
        'path': 's6446008.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-11',
        'path': 's6848399.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
]

# --- Group 2D: Pure PR, k-Dyck Truncated Shuffled - D4, late-to-early ---
json_map_group2d_late = [
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L0-11',
        'path': 's6848399.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L1-11',
        'path': 's6463436.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L2-11',
        'path': 's6463393.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L3-11',
        'path': 's6463397.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L4-11',
        'path': 's6463400.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L5-11',
        'path': 's6463401.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L6-11',
        'path': 's6463402.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L7-11',
        'path': 's6463403.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L8-11',
        'path': 's6463404.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L9-11',
        'path': 's6463405.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L10-11',
        'path': 's6463406.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L11',
        'path': 's6463407.json',
        'init': 'k-Dyck Truncated Shuffled - D4',
    },
]

# --- Group 2E: Pure PR, k-Dyck - D98 ---
json_map_group2e_late = [
    {
        'category': 'Procedural-init',
        'layer': 'L0-11',
        'path': 's6848383.json',
        'init': 'k-Dyck - D98',
    }
]

# L0-11 is the shared convergence point for both sweep directions (see the early/late
# classification rule above), so the same single entry is also the 'early' leaf.
json_map_group2e_early = json_map_group2e_late

# --- Group 2F: Pure PR, k-Dyck Truncated - D4 ---
json_map_group2f_late = [
    {
        'category': 'Procedural-init',
        'layer': 'L0-11',
        'path': 's6848389.json',
        'init': 'k-Dyck Truncated - D4',
    }
]

json_map_group2f_early = json_map_group2f_late

# --- Group 2G: Pure PR, k-Dyck Shuffled - D4 ---
json_map_group2g_late = [
    {
        'category': 'Procedural-init',
        'layer': 'L0-11',
        'path': 's6848393.json',
        'init': 'k-Dyck Shuffled - D4',
    }
]

json_map_group2g_early = json_map_group2g_late

# --- Group 2H: Pure PR, k-Dyck - D98 ---
json_map_group2h_late = [
    {
        'category': 'Procedural-init',
        'layer': 'L0-11',
        'path': 's6848400.json',
        'init': 'k-Dyck Truncated Shuffled - D98',
    }
]

json_map_group2h_early = json_map_group2h_late

# --- Group 3A: PR-init weights + Rand-init norm ratios, k-Dyck - D4, early-to-late ---
json_map_group3a_early = [
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0',
        'path': 's6743897.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-1',
        'path': 's6743899.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-2',
        'path': 's6743900.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-3',
        'path': 's6743901.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-4',
        'path': 's6743905.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-5',
        'path': 's6743906.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-6',
        'path': 's6743907.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-7',
        'path': 's6743908.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-8',
        'path': 's6743909.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-9',
        'path': 's6743910.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-10',
        'path': 's6743911.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-11',
        'path': 's6191465.json',
        'init': 'k-Dyck - D4',
    },
]

# --- Group 3A: PR-init weights + Rand-init norm ratios, k-Dyck - D4, late-to-early ---
json_map_group3a_late = [
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-11',
        'path': 's6191465.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L1-11',
        'path': 's6221477.json',
        'init': 'k-Dyck - D4',
    },
    # { rerun
    #     'category': 'PR-init w/ Rand-init Norm Ratios',
    #     'layer': 'L2-11',
    #     'path': 's6773623.json',
    #     'init': 'k-Dyck - D4',
    # },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L3-11',
        'path': 's6773624.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L4-11',
        'path': 's6773625.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L5-11',
        'path': 's6773628.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L6-11',
        'path': 's6202749.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L7-11',
        'path': 's6773631.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L8-11',
        'path': 's6773632.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L9-11',
        'path': 's6221557.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L10-11',
        'path': 's6794033.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L11',
        'path': 's6773635.json',
        'init': 'k-Dyck - D4',
    },
]

# --- Group 3B: PR-init weights + Rand-init norm ratios, k-Dyck Shuffled - D98, early-to-late ---
json_map_group3b_early = [
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0',
        'path': 's6743625.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-1',
        'path': 's6743626.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-2',
        'path': 's6743627.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-3',
        'path': 's6743630.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-4',
        'path': 's6743631.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-5',
        'path': 's6743632.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-6',
        'path': 's6743633.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-7',
        'path': 's6743634.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-8',
        'path': 's6743635.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-9',
        'path': 's6743636.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-10',
        'path': 's6743637.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-11',
        'path': 's6225011.json',
        'init': 'k-Dyck Shuffled - D98',
    },
]

# --- Group 3B: PR-init weights + Rand-init norm ratios, k-Dyck Shuffled - D98, late-to-early ---
json_map_group3b_late = [
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-11',
        'path': 's6225011.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L1-11',
        'path': 's6736970.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L2-11',
        'path': 's6736971.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L3-11',
        'path': 's6736975.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L4-11',
        'path': 's6736972.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L5-11',
        'path': 's6736973.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L6-11',
        'path': 's6736974.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L7-11',
        'path': 's6736976.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L8-11',
        'path': 's6736977.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L9-11',
        'path': 's6736978.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L10-11',
        'path': 's6736979.json',
        'init': 'k-Dyck Shuffled - D98',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L11',
        'path': 's6736980.json',
        'init': 'k-Dyck Shuffled - D98',
    },
]

# --- Group 4A: ViT-Base Pure PR, early-to-late ---
json_map_group4a_early = [
    {
        'category': '',
        'layer': '',
        'path': 'accuracy_IMNET_BASE_29384839.json',
        'init': 'Random-init',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0',
        'path': 'accuracy_IMNET_BASE_29457108.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-1',
        'path': 'accuracy_IMNET_BASE_29462316.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-10',
        'path': 'accuracy_IMNET_BASE_29469074.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-2',
        'path': 'accuracy_IMNET_BASE_29462317.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-3',
        'path': 'accuracy_IMNET_BASE_29457107.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-3',
        'path': 'accuracy_IMNET_BASE_29469064.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-3',
        'path': 'accuracy_IMNET_BASE_29469063.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-4',
        'path': 'accuracy_IMNET_BASE_29457109.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-4',
        'path': 'accuracy_IMNET_BASE_29469067.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-4',
        'path': 'accuracy_IMNET_BASE_29469068.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-5',
        'path': 'accuracy_IMNET_BASE_29451646.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-6',
        'path': 'accuracy_IMNET_BASE_29451645.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-7',
        'path': 'accuracy_IMNET_BASE_29448854.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-8',
        'path': 'accuracy_IMNET_BASE_29469072.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Procedural-init / Random-init',
        'layer': 'L0-9',
        'path': 'accuracy_IMNET_BASE_29469073.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR',
        'layer': 'L0-11',
        'path': 'accuracy_IMNET_BASE_29377576.json',
        'init': 'k-Dyck - D4',
    },
]

# --- Group 4A: ViT-Base Pure PR, late-to-early ---
json_map_group4a_late = [
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L1-11',
        'path': 'accuracy_IMNET_BASE_29484978.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L2-11',
        'path': 'accuracy_IMNET_BASE_29484977.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L11',
        'path': 'accuracy_IMNET_BASE_29484973.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L3-11',
        'path': 'accuracy_IMNET_BASE_29484976.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L4-11',
        'path': 'accuracy_IMNET_BASE_29484975.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L5-11',
        'path': 'accuracy_IMNET_BASE_29469076.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L6-11',
        'path': 'accuracy_IMNET_BASE_29457111.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L7-11',
        'path': 'accuracy_IMNET_BASE_29457110.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L8-11',
        'path': 'accuracy_IMNET_BASE_29448853.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L9-11',
        'path': 'accuracy_IMNET_BASE_29469075.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random-init / Procedural-init',
        'layer': 'L10-11',
        'path': 'accuracy_IMNET_BASE_29484974.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR',
        'layer': 'L0-11',
        'path': 'accuracy_IMNET_BASE_29377576.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': '',
        'layer': '',
        'path': 'accuracy_IMNET_BASE_29384839.json',
        'init': 'Random-init',
    },
]

# --- Group 4B: ViT-Base PR-init weights + Rand-init norm ratios, early-to-late ---
json_map_group4b_early = [
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0',
        'path': 'accuracy_IMNET_BASE_29514777.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-7',
        'path': 'accuracy_IMNET_BASE_29451650.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-11',
        'path': 'accuracy_IMNET_BASE_29413395.json',
        'init': 'k-Dyck - D4',
    },
]

# --- Group 4B: ViT-Base PR-init weights + Rand-init norm ratios, late-to-early ---
json_map_group4b_late = [
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L0-11',
        'path': 'accuracy_IMNET_BASE_29413395.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L1-11',
        'path': 'accuracy_IMNET_BASE_29520494.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L10-11',
        'path': 'accuracy_IMNET_BASE_29405954.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L11',
        'path': 'accuracy_IMNET_BASE_29408586.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L2-11',
        'path': 'accuracy_IMNET_BASE_29520493.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L3-11',
        'path': 'accuracy_IMNET_BASE_29520492.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L4-11',
        'path': 'accuracy_IMNET_BASE_29409056.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L5-11',
        'path': 'accuracy_IMNET_BASE_29520491.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L6-11',
        'path': 'accuracy_IMNET_BASE_29409054.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L7-11',
        'path': 'accuracy_IMNET_BASE_29408996.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L8-11',
        'path': 'accuracy_IMNET_BASE_29407013.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'PR-init w/ Rand-init Norm Ratios',
        'layer': 'L9-11',
        'path': 'accuracy_IMNET_BASE_29407014.json',
        'init': 'k-Dyck - D4',
    },
]

# --- Group 4C: ViT-Base Rand-init weights + PR-init norm ratios, early-to-late ---
json_map_group4c_early = [
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L0',
        'path': 'accuracy_IMNET_BASE_29514778.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L0-7',
        'path': 'accuracy_IMNET_BASE_29451652.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L0-8',
        'path': 'accuracy_IMNET_BASE_29538140.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L0-11',
        'path': 'accuracy_IMNET_BASE_29451648.json',
        'init': 'k-Dyck - D4',
    },
]

# --- Group 4C: ViT-Base Rand-init weights + PR-init norm ratios, late-to-early ---
json_map_group4c_late = [
    {
        'category': '',
        'layer': '',
        'path': 'accuracy_IMNET_BASE_29384839.json',
        'init': 'Random-init',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L0-11',
        'path': 'accuracy_IMNET_BASE_29451648.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L1-11',
        'path': 'accuracy_IMNET_BASE_29520497.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L10-11',
        'path': 'accuracy_IMNET_BASE_29388197.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L10-11',
        'path': 'accuracy_IMNET_BASE_29406781.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L10-11',
        'path': 'accuracy_IMNET_BASE_29406780.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L11',
        'path': 'accuracy_IMNET_BASE_29388181.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L2-11',
        'path': 'accuracy_IMNET_BASE_29520496.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L3-11',
        'path': 'accuracy_IMNET_BASE_29520495.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L4-11',
        'path': 'accuracy_IMNET_BASE_29405945.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L5-11',
        'path': 'accuracy_IMNET_BASE_29405944.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L6-11',
        'path': 'accuracy_IMNET_BASE_29388247.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L7-11',
        'path': 'accuracy_IMNET_BASE_29388227.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L8-11',
        'path': 'accuracy_IMNET_BASE_29388254.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L8-11',
        'path': 'accuracy_IMNET_BASE_29406776.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L8-11',
        'path': 'accuracy_IMNET_BASE_29406777.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L9-11',
        'path': 'accuracy_IMNET_BASE_29388202.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L9-11',
        'path': 'accuracy_IMNET_BASE_29406779.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Rand-init w/ PR-init Norm Ratios',
        'layer': 'L9-11',
        'path': 'accuracy_IMNET_BASE_29406778.json',
        'init': 'k-Dyck - D4',
    },
]


json_maps_by_subset = {
    "1A": json_map_group1a,
    "1B": json_map_group1b,
    "2A": {"early": json_map_group2a_early, "late": json_map_group2a_late},
    "2B": {"early": json_map_group2b_early, "late": json_map_group2b_late},
    "2C": {"early": json_map_group2c_early, "late": json_map_group2c_late},
    "2D": {"early": json_map_group2d_early, "late": json_map_group2d_late},
    # 2E-2H: single-checkpoint (L0-11 only) Pure-PR additions for the 4 k-Dyck variants that
    # don't have a full layer-scaling sweep. L0-11 is the shared early/late convergence point,
    # so the same entry is registered as both directions' leaf.
    "2E": {"early": json_map_group2e_early, "late": json_map_group2e_late},
    "2F": {"early": json_map_group2f_early, "late": json_map_group2f_late},
    "2G": {"early": json_map_group2g_early, "late": json_map_group2g_late},
    "2H": {"early": json_map_group2h_early, "late": json_map_group2h_late},
    "3A": {"early": json_map_group3a_early, "late": json_map_group3a_late},
    "3B": {"early": json_map_group3b_early, "late": json_map_group3b_late},
    "4A": {"early": json_map_group4a_early, "late": json_map_group4a_late},
    "4B": {"early": json_map_group4b_early, "late": json_map_group4b_late},
    "4C": {"early": json_map_group4c_early, "late": json_map_group4c_late},
}

for _subset, _data in json_maps_by_subset.items():
    if isinstance(_data, dict):
        print(f"{_subset}: early={len(_data['early'])} late={len(_data['late'])}")
    else:
        print(f"{_subset}: {len(_data)} entries")


In [ ]:
# --- Custom: hand-provided list of checkpoint JSON entries, registered as its own subset ---
# Fill in one dict per checkpoint file, same shape as the other groups above:
#   category -- free-text label used to build the plotted "label" column
#   layer    -- which layers were trained/scaled for this checkpoint (display only)
#   path     -- filename of the checkpoint's JSON, relative to `parent_path`
#   init     -- init/dataset tag used to build the plotted "label" column
json_map_custom = [
    {'init': 'Random-init', 'path': 's6257433.json'},
    {"init": "k-Dyck - D4", "category": "PR", "layer": "L0-11", "path": "s6848384.json"},
    {"init": "k-Dyck - D98", "category": "PR", "layer": "L0-11", "path": "s6848383.json"},  # new (was s4922828.json)
    {"init": "k-Dyck Truncated - D4", "category": "PR", "layer": "L0-11", "path": "s6848389.json"},  # new (was s4922830.json)
    {"init": "k-Dyck Truncated - D98", "category": "PR", "layer": "L0-11", "path": "s6848392.json"},
    {"init": "k-Dyck Shuffled - D4", "category": "PR", "layer": "L0-11", "path": "s6848393.json"},  # new (was s4922832.json)
    {"init": "k-Dyck Shuffled - D98", "category": "PR", "layer": "L0-11", "path": "s6848396.json"},
    {"init": "k-Dyck Truncated Shuffled - D4", "category": "PR", "layer": "L0-11", "path": "s6848399.json"},
    {"init": "k-Dyck Truncated Shuffled - D98", "category": "PR", "layer": "L0-11", "path": "s6848400.json"}
  ]

print([a["path"].split(".")[0][1:] for a in json_map_custom])

json_maps_by_subset["CUSTOM"] = json_map_custom
print(f"CUSTOM: {len(json_map_custom)} entries")

json_maps_by_subset["VITB_BASELINE"] = [
    # {
    #     "init": "k-Dyck - D4",
    #     "category": "PR",
    #     "layer": "L0-8",
    #     "path": "s6316138.json",
    # },
    # {
    #     "init": "Random-init",
    #     "category": "",
    #     "layer": "",
    #     "path": "6257433",
    # },
    {
        'category': '',
        'layer': '',
        'path': 'accuracy_IMNET_BASE_29384839.json',
        'init': 'Random init',
    },
    {
        'category': '',
        'layer': '',
        'path': 'accuracy_IMNET_BASE_29377576.json',
        'init': 'k-Dyck - D4',
    },
    {
        "init": "k-Dyck Shuffled - D98",
        "category": "",
        "path": "ftb4_d98_accuracy_IMNET_BASE_29547831.json"
    },
]
json_maps_by_subset["VITB_ANALYTICAL"] = [
    {
        'category': 'Random init w/ PW properties',
        'layer': '',
        'path': 'accuracy_IMNET_BASE_29737095_s0.json',
        'init': 'k-Dyck - D4',
    },
    {
        'category': 'Random init w/ PW properties',
        'layer': '',
        'path': 'accuracy_IMNET_BASE_29736861_s0.json',
        'init': 'k-Dyck Shuffled - D98',
    },
]


## Groups 5 & 6: cross-dataset pooled categories, plus Groups 0 / 2-master / 3-master

Group 5 pools the k-Dyck datasets together per category x direction (rather than splitting by dataset like Groups 2/3):

- **5A / 5B**: Pure PR (PR-init weights + PR-init norm ratios), pooled across all 8 k-Dyck datasets Group 2 now covers -- the original 4 with a full layer-scaling sweep (D4, Shuffled-D98, Truncated-D98, Truncated-Shuffled-D4, Groups `2A`-`2D`) plus the 4 single-checkpoint additions (D98, Truncated-D4, Shuffled-D4, Truncated-Shuffled-D98, Groups `2E`-`2H`) -- early-to-late / late-to-early.
- **5C / 5D**: PR-init weights + Rand-init norm ratios, pooled across the 2 datasets this scaling ablation exists for (D4, Shuffled-D98) -- early-to-late / late-to-early.
- **5E / 5F**: Rand-init weights + PR-init norm ratios, pooled across D4 + Shuffled-D98 -- early-to-late / late-to-early. This is the same category/direction/dataset combination as Group 1's `1A`/`1B`, so `5E`/`5F` are the same data (kept as separate named subsets for clarity/traceability).

Group 6 pools all three Group-5 categories together, by direction only:

- **6A**: everything early-to-late from `5A` + `5C` + `5E`.
- **6B**: everything late-to-early from `5B` + `5D` + `5F`.

Built directly from the already-split `json_map_group*` lists above (no new master-map parsing needed), deduplicated by `path`. Added to `json_maps_by_subset` as flat lists (like `1A`/`1B`), since direction is already baked into the letter.

**Groups 2E-2H** are new: each is a single L0-11 (fully-scaled) Pure-PR checkpoint for a k-Dyck variant that doesn't have a partial layer-scaling sweep. Since L0-11 is the shared convergence point both sweep directions converge on (the same rule used everywhere else in this notebook), the same single entry is registered as both leaves' data (`2E_early == 2E_late`, etc.) rather than living in only one direction.

**Dynamic pooling.** `pool_leaf_subsets(json_maps_by_subset, leaf_names)` takes any list of already-flattened leaf names (e.g. `["2A_early", "3B_late"]`) and returns one deduplicated pooled `json_map` -- so an ad-hoc grouping can be built on the fly from any combination of leaves without hand-coding a new named list for it every time. Three convenience groups are built this way and registered into `json_maps_by_subset`:

- **`0`**: every leaf under ViT-Small (`imnet100_small`) -- i.e. everything from Groups 1, 2 (including `2E`-`2H`), 3, 5, and 6, pooled into one deduplicated subset. Group 4 (ViT-Base / `imnet_base`) is excluded.
- **`2_master`**: every Group-2 leaf (`2A`-`2H`, both directions) pooled into one -- all Pure-PR data across all 8 k-Dyck variants and both sweep directions combined.
- **`3_master`**: every Group-3 leaf (`3A`, `3B`, both directions) pooled into one.

Both `0`, `2_master`, and `3_master` are flat (not split by direction), since they're deliberately pooling across both directions.


In [ ]:
def _dedup_by_path(entries):
    seen = set()
    out = []
    for e in entries:
        if e["path"] in seen:
            continue
        seen.add(e["path"])
        out.append(e)
    return out


def flatten_subsets(json_maps_by_subset):
    """Flatten {'1A': [...], '2A': {'early':[...],'late':[...]}, ...} into leaf_name -> json_map."""
    flat = {}
    for subset, data in json_maps_by_subset.items():
        if isinstance(data, dict):
            for direction, jm in data.items():
                flat[f"{subset}_{direction}"] = jm
        else:
            flat[subset] = data
    return flat


# --- Group 5: cross-dataset pooled scaling categories (all 8 k-Dyck datasets combined per direction) ---
# 5A / 5B: Pure PR (PR-init weights + PR-init norm ratios), pooled across every k-Dyck dataset
# with Pure-PR data: the 4 with a full layer-scaling sweep (D4, Shuffled-D98, Truncated-D98,
# Truncated-Shuffled-D4, from Groups 2A-2D) plus the 4 single-checkpoint additions (D98,
# Truncated-D4, Shuffled-D4, Truncated-Shuffled-D98, from Groups 2E-2H).
json_map_group5a = _dedup_by_path(
    json_map_group2a_early + json_map_group2b_early + json_map_group2c_early + json_map_group2d_early
    + json_map_group2e_early + json_map_group2f_early + json_map_group2g_early + json_map_group2h_early
)
json_map_group5b = _dedup_by_path(
    json_map_group2a_late + json_map_group2b_late + json_map_group2c_late + json_map_group2d_late
    + json_map_group2e_late + json_map_group2f_late + json_map_group2g_late + json_map_group2h_late
)

# 5C / 5D: PR-init weights + Rand-init norm ratios, pooled across the datasets this scaling
# ablation was run on (D4 + Shuffled-D98 only; see Group 3's note).
json_map_group5c = _dedup_by_path(json_map_group3a_early + json_map_group3b_early)
json_map_group5d = _dedup_by_path(json_map_group3a_late + json_map_group3b_late)

# 5E / 5F: Rand-init weights + PR-init norm ratios, pooled across D4 + Shuffled-D98. This is the
# same category/direction/dataset combination as Group 1's 1A/1B, so it's the same data.
json_map_group5e = json_map_group1a
json_map_group5f = json_map_group1b

# --- Group 6: all three Group-5 categories pooled together, by direction ---
json_map_group6a = _dedup_by_path(json_map_group5a + json_map_group5c + json_map_group5e)
json_map_group6b = _dedup_by_path(json_map_group5b + json_map_group5d + json_map_group5f)

json_maps_by_subset["5A"] = json_map_group5a
json_maps_by_subset["5B"] = json_map_group5b
json_maps_by_subset["5C"] = json_map_group5c
json_maps_by_subset["5D"] = json_map_group5d
json_maps_by_subset["5E"] = json_map_group5e
json_maps_by_subset["5F"] = json_map_group5f
json_maps_by_subset["6A"] = json_map_group6a
json_maps_by_subset["6B"] = json_map_group6b

for _subset in ["5A", "5B", "5C", "5D", "5E", "5F", "6A", "6B"]:
    print(f"{_subset}: {len(json_maps_by_subset[_subset])} entries")


def pool_leaf_subsets(json_maps_by_subset, leaf_names, dedup=True):
    """Dynamically pool one or more already-flattened leaf subsets (e.g. '2A_early', '3B_late',
    or a flat subset name like '1A') into a single json_map, by name -- so an ad-hoc grouping
    can be built on the fly from any combination of existing leaves without hand-coding a new
    named list/variable for it. Raises KeyError (listing what IS available) on a typo'd name."""
    flat = flatten_subsets(json_maps_by_subset)
    pooled = []
    for name in leaf_names:
        if name not in flat:
            raise KeyError(f"Unknown leaf subset: {name!r}. Available: {sorted(flat)}")
        pooled.extend(flat[name])
    return _dedup_by_path(pooled) if dedup else pooled


# --- Group 0: every leaf under ViT-Small (imnet100_small) -- i.e. everything from Groups
# 1, 2 (incl. 2E-2H), 3, 5, and 6 pooled into one, deduplicated by path. Group 4 (prefix "4")
# is ViT-Base (imnet_base), so it's excluded here.
_group0_leaves = [name for name in flatten_subsets(json_maps_by_subset) if not _is_vitb_leaf(name)]
json_map_group0 = pool_leaf_subsets(json_maps_by_subset, _group0_leaves)
json_maps_by_subset["0"] = json_map_group0

# --- Group 2 master: every Group-2 leaf (2A-2H, both directions) pooled into one flat subset --
# i.e. all Pure-PR data across all 8 k-Dyck variants and both sweep directions combined.
_group2_leaves = [name for name in flatten_subsets(json_maps_by_subset) if name.startswith("2")]
json_map_group2_master = pool_leaf_subsets(json_maps_by_subset, _group2_leaves)
json_maps_by_subset["2_master"] = json_map_group2_master

# --- Group 3 master: every Group-3 leaf (3A, 3B, both directions) pooled into one flat subset.
_group3_leaves = [name for name in flatten_subsets(json_maps_by_subset) if name.startswith("3")]
json_map_group3_master = pool_leaf_subsets(json_maps_by_subset, _group3_leaves)
json_maps_by_subset["3_master"] = json_map_group3_master

# --- Group 4 master: every Group-4 leaf (4A-4C, both directions) pooled into one flat subset --
# i.e. all ViT-Base (imnet_base) data across all three category patterns and both directions.
_group4_leaves = [name for name in flatten_subsets(json_maps_by_subset) if _is_vitb_leaf(name)]
json_map_group4_master = pool_leaf_subsets(json_maps_by_subset, _group4_leaves)
json_maps_by_subset["4_master"] = json_map_group4_master

# --- Group 4 early/late: every Group-4 leaf pooled by direction only (early across 4A/4B/4C,
# late across 4A/4B/4C), mirroring Group 4 master's "ignore category, keep direction" split.
json_map_group4_early = pool_leaf_subsets(json_maps_by_subset, ["4A_early", "4B_early", "4C_early"])
json_map_group4_late = pool_leaf_subsets(json_maps_by_subset, ["4A_late", "4B_late", "4C_late"])
json_maps_by_subset["4_early"] = json_map_group4_early
json_maps_by_subset["4_late"] = json_map_group4_late

_master_leaf_counts = {
    "0": _group0_leaves, "2_master": _group2_leaves, "3_master": _group3_leaves, "4_master": _group4_leaves,
    "4_early": ["4A_early", "4B_early", "4C_early"], "4_late": ["4A_late", "4B_late", "4C_late"],
}
for _subset, _leaves in _master_leaf_counts.items():
    print(f"{_subset}: {len(json_maps_by_subset[_subset])} entries (pooled from {len(_leaves)} leaves)")



## Group 7: General (four PR/Rand-init layer-mixing patterns, all 8 k-Dyck datasets)

`report_plots.ipynb`'s `## general` section (the `full_pr` set) defines, for each of the 8 k-Dyck datasets, 4 active layer-mixing patterns plus a standalone Random-init baseline:

- **7A -- All PR**: category `PR`, no partial layer split (the whole 12-layer network is Pure-PR) -- equivalent to `L0-11` in this notebook's convention.
- **7B -- PR 0-8**: category `PR`, layer `L0-8` (layers 0-8 Pure-PR, 9-11 left at whatever the base checkpoint init is).
- **7C -- PR 0-8 + Rand-init w/ PR-init Norm Ratios**: layers 0-8 Pure-PR, layers 9-11 Rand-init weights with PR-init norm ratios.
- **7D -- PR 0-8 + PR-init w/ Rand-init Norm Ratios**: layers 0-8 Pure-PR, layers 9-11 PR-init weights with Rand-init norm ratios.

**Path sourcing:** for `7B`-`7D` and the Random-init-mixing patterns in general, the file paths are copied as-is from `report_plots.ipynb` (bare numeric paths there get the same `s<id>.json` normalization). For **7A (All PR)** and the **standalone Random-init baseline**, `delta_norm_epoch_analysis.ipynb` has newer checkpoints for some of these runs than `report_plots.ipynb` did, and those newer ones are used instead:
- Random-init baseline: `s6257433.json` (new) replaces `report_plots.ipynb`'s `s4912449.json`.
- All-PR: `D98`, `Truncated - D4`, `Shuffled - D4`, and `Truncated Shuffled - D98` have newer L0-11 checkpoints already registered as this notebook's `2E`-`2H` (`s6848383.json`, `s6848389.json`, `s6848393.json`, `s6848400.json`); the other 4 datasets (`D4`, `Truncated - D98`, `Shuffled - D98`, `Truncated Shuffled - D4`) have no newer replacement, so `report_plots.ipynb`'s original paths (`s4739585.json`, `s4992673.json`, `s4992676.json`, `s4992678.json`) are kept.

`7A`-`7D` are registered as flat subsets (there's no early/late sweep direction here -- each is a fixed, single-checkpoint-per-dataset pattern). `7_master` pools all four types plus the Random-init baseline into one deduplicated group, following the same `pool_leaf_subsets`/`_dedup_by_path` pattern as `0` / `2_master` / `3_master`.


In [ ]:
# --- Group 7A: All PR (full 12-layer Pure-PR, no partial layer split == "L0-11") ---
json_map_group7a = [
    {'init': 'Random-init', 'path': 's6257433.json'},
    {"init": "k-Dyck - D4", "category": "PR", "layer": "L0-11", "path": "s6848384.json"},
    {"init": "k-Dyck - D98", "category": "PR", "layer": "L0-11", "path": "s6848383.json"},  # new (was s4922828.json)
    {"init": "k-Dyck Truncated - D4", "category": "PR", "layer": "L0-11", "path": "s6848389.json"},  # new (was s4922830.json)
    {"init": "k-Dyck Truncated - D98", "category": "PR", "layer": "L0-11", "path": "s6848392.json"},
    {"init": "k-Dyck Shuffled - D4", "category": "PR", "layer": "L0-11", "path": "s6848393.json"},  # new (was s4922832.json)
    {"init": "k-Dyck Shuffled - D98", "category": "PR", "layer": "L0-11", "path": "s6848396.json"},
    {"init": "k-Dyck Truncated Shuffled - D4", "category": "PR", "layer": "L0-11", "path": "s6848399.json"},
    {"init": "k-Dyck Truncated Shuffled - D98", "category": "PR", "layer": "L0-11", "path": "s6848400.json"},  # new (was s4912446.json)
]

# --- Group 7B: PR 0-8 ---
json_map_group7b = [
    {"init": "k-Dyck - D4", "category": "PR", "layer": "L0-8", "path": "s6316138.json"},
    {"init": "k-Dyck - D98", "category": "PR", "layer": "L0-8", "path": "s6759125.json"},
    {"init": "k-Dyck Truncated - D4", "category": "PR", "layer": "L0-8", "path": "s6759127.json"},
    {"init": "k-Dyck Truncated - D98", "category": "PR", "layer": "L0-8", "path": "s6379252.json"},
    {"init": "k-Dyck Shuffled - D4", "category": "PR", "layer": "L0-8", "path": "s6759130.json"},
    {"init": "k-Dyck Shuffled - D98", "category": "PR", "layer": "L0-8", "path": "s6329840.json"},
    {"init": "k-Dyck Truncated Shuffled - D4", "category": "PR", "layer": "L0-8", "path": "s6446006.json"},
    {"init": "k-Dyck Truncated Shuffled - D98", "category": "PR", "layer": "L0-8", "path": "s6759132.json"},
]

# --- Group 7C: PR 0-8 + Rand-init w/ PR-init Norm Ratios (layers 9-11) ---
json_map_group7c = [
    {"init": "k-Dyck - D4", "category": "PR L0-8 + Rand-init w/ PR-init Norm Ratios", "layer": "L9-11", "path": "s6195717.json"},
    {"init": "k-Dyck - D98", "category": "PR L0-8 + Rand-init w/ PR-init Norm Ratios", "layer": "L9-11", "path": "s6759727.json"},
    {"init": "k-Dyck Truncated - D4", "category": "PR L0-8 + Rand-init w/ PR-init Norm Ratios", "layer": "L9-11", "path": "s6759728.json"},
    {"init": "k-Dyck Truncated - D98", "category": "PR L0-8 + Rand-init w/ PR-init Norm Ratios", "layer": "L9-11", "path": "s6759729.json"},
    {"init": "k-Dyck Shuffled - D4", "category": "PR L0-8 + Rand-init w/ PR-init Norm Ratios", "layer": "L9-11", "path": "s6759730.json"},
    {"init": "k-Dyck Shuffled - D98", "category": "PR L0-8 + Rand-init w/ PR-init Norm Ratios", "layer": "L9-11", "path": "s6759731.json"},
    {"init": "k-Dyck Truncated Shuffled - D4", "category": "PR L0-8 + Rand-init w/ PR-init Norm Ratios", "layer": "L9-11", "path": "s6759734.json"},
    {"init": "k-Dyck Truncated Shuffled - D98", "category": "PR L0-8 + Rand-init w/ PR-init Norm Ratios", "layer": "L9-11", "path": "s6759735.json"},
]

# --- Group 7D: PR 0-8 + PR-init w/ Rand-init Norm Ratios (layers 9-11) ---
json_map_group7d = [
    {"init": "k-Dyck - D4", "category": "PR L0-8 + PR-init w/ Rand-init Norm Ratios", "layer": "L9-11", "path": "s6790001.json"},
    {"init": "k-Dyck - D98", "category": "PR L0-8 + PR-init w/ Rand-init Norm Ratios", "layer": "L9-11", "path": "s6794082.json"},
    {"init": "k-Dyck Truncated - D4", "category": "PR L0-8 + PR-init w/ Rand-init Norm Ratios", "layer": "L9-11", "path": "s6794094.json"},
    {"init": "k-Dyck Truncated - D98", "category": "PR L0-8 + PR-init w/ Rand-init Norm Ratios", "layer": "L9-11", "path": "s6794096.json"},
    {"init": "k-Dyck Shuffled - D4", "category": "PR L0-8 + PR-init w/ Rand-init Norm Ratios", "layer": "L9-11", "path": "s6794097.json"},
    {"init": "k-Dyck Shuffled - D98", "category": "PR L0-8 + PR-init w/ Rand-init Norm Ratios", "layer": "L9-11", "path": "s6794098.json"},
    {"init": "k-Dyck Truncated Shuffled - D4", "category": "PR L0-8 + PR-init w/ Rand-init Norm Ratios", "layer": "L9-11", "path": "s6804646.json"},
    {"init": "k-Dyck Truncated Shuffled - D98", "category": "PR L0-8 + PR-init w/ Rand-init Norm Ratios", "layer": "L9-11", "path": "s6794103.json"},
]

# --- Standalone Random-init baseline (new path from delta_norm_epoch_analysis.ipynb) ---
json_map_group7_randinit = [
    {"init": "Random-init", "category": "", "layer": "", "path": "s6257433.json"},  # new (was s4912449.json)
]

json_maps_by_subset["7A"] = json_map_group7a
json_maps_by_subset["7B"] = json_map_group7b
json_maps_by_subset["7C"] = json_map_group7c
json_maps_by_subset["7D"] = json_map_group7d

# --- Group 7 master: all four patterns + the Random-init baseline, pooled into one ---
json_map_group7_master = _dedup_by_path(
    json_map_group7a + json_map_group7b + json_map_group7c + json_map_group7d + json_map_group7_randinit
)
json_maps_by_subset["7_master"] = json_map_group7_master

for _subset in ["7A", "7B", "7C", "7D", "7_master"]:
    print(f"{_subset}: {len(json_maps_by_subset[_subset])} entries")


## Group 8: split by layer half (layers 0-5 vs layers 6-11)

Splits the same pool used for Group `0` (every ViT-Small scaling/procedural-weights leaf -- Groups 1, 2 incl. `2E`-`2H`, 3, 5, and 6) into two flat, direction-agnostic subsets by which half of the network the entry's `layer` range falls in:

- **8A**: layers 0-5 -- entries whose `layer` string is fully contained within `L0`-`L5` (e.g. `L0`, `L0-3`, `L0-5`).
- **8B**: layers 6-11 -- entries whose `layer` string is fully contained within `L6`-`L11` (e.g. `L6-11`, `L9-11`, `L11`).

**Inclusion rule (fully-contained only):** an entry is classified into a half only if its *entire* layer range lies within that half. Entries that straddle the 5/6 boundary (`L0-8`, `L3-11`, `L0-11`, etc.) are excluded from *both* 8A and 8B, since neither cleanly represents "layers 0-5 changed" or "layers 6-11 changed" alone. Entries with no layer range at all (e.g. the plain Random-init baseline) are also excluded, since they carry no scaling/procedural-weight change to attribute to either half.

Built by parsing each `json_map_group0` entry's `layer` field (format `L<start>-<end>` or `L<n>`) and filtering -- no new master-map parsing needed, same as Groups 5/6.


In [ ]:
import re


def _parse_layer_range(layer_str):
    """'L0-8' -> (0, 8); 'L11' -> (11, 11); '' or None -> None."""
    if not layer_str:
        return None
    m = re.fullmatch(r"L(\d+)(?:-(\d+))?", layer_str)
    if not m:
        return None
    start = int(m.group(1))
    end = int(m.group(2)) if m.group(2) is not None else start
    return start, end


def _fully_in_range(layer_str, lo, hi):
    parsed = _parse_layer_range(layer_str)
    if parsed is None:
        return False
    start, end = parsed
    return lo <= start and end <= hi


# --- Group 8A: layers 0-5 (entries fully contained within L0-L5) ---
json_map_group8a = [e for e in json_map_group0 if _fully_in_range(e["layer"], 0, 5)]

# --- Group 8B: layers 6-11 (entries fully contained within L6-L11) ---
json_map_group8b = [e for e in json_map_group0 if _fully_in_range(e["layer"], 6, 11)]

json_maps_by_subset["8A"] = json_map_group8a
json_maps_by_subset["8B"] = json_map_group8b

for _subset in ["8A", "8B"]:
    print(f"{_subset}: {len(json_maps_by_subset[_subset])} entries")


## Task 3: Spearman ρ heatmap (masked by significance), per group

For each requested group (by default just `2A_early`, matching the Task 2 smoke test above -- see the call at the bottom of the cell for the full-sweep list), this builds, per model and per transformer component (`layer 0-11 x attn/mlp`, 24 components), six norm-ratio features from the `delta_norm_ratio` training series:

1. `init` -- the one-off `custom_delta_norm_ratio` value logged at epoch −1 (before any training delta exists; distinct from the ongoing `delta_norm_ratio` series).
2. `0-50`, `50-100`, `100-200`, `200-299` -- the **median** `delta_norm_ratio` within each Task-1 epoch bucket.
3. `e49` -- the `delta_norm_ratio` value at the epoch whose actual epoch value is 49.

Each (model, component) row then has these 6 features plus `final_accuracy`. For every component, across all models in the group, this computes **Spearman's ρ** (rank correlation) and its p-value between each feature and `final_accuracy`.

Results are cached to `<group>_task3_features.csv` (raw per-model features), `<group>_task3_spearman.csv` / `<group>_task3_spearman_pvalue.csv` (the 24x6 matrices), a `<group>_task3_spearman_heatmap.json` with the same matrix values (for reprocessing into a table or re-plotting later), and the heatmap itself (`<group>_task3_spearman_heatmap.png`/`.pdf`) -- x-axis = feature, y-axis = the 24 components, seaborn's diverging `coolwarm` colormap centered at 0. Cells whose Spearman p-value is not significant (p ≥ 0.05, or undefined) are faded out with a translucent white overlay, so validity reads visually instead of requiring a second look-up. Figure sizing comes from the `tueplots` `bundles.iclr2024` config set in the first cell.

Groups are looked up by name in `json_maps_by_subset` after flattening (any leaf like `2A_early`, or one of the pooled subsets `0` / `2_master` / `3_master` / `5A`-`6B`); an arbitrary ad-hoc combination of leaves can also be passed as a `(group_name, [leaf_names])` tuple, pooled on the fly via `pool_leaf_subsets`.


In [ ]:
OUTPUT_DIR_TASK3 = OUTPUT_DIR  # reuse the scaling_hpothesis output dir from cell above


def process_jsons_training_with_init(json_map, parent_path):
    """Like process_jsons_training, but also pulls the one-off "custom_delta_norm_ratio"
    init value (epoch -1) per layer/type into "{col}_init" columns."""
    all_data = []
    for json_item in json_map:
        json_path = json_item["path"]
        file_path = os.path.join(parent_path, json_path)
        if not os.path.exists(file_path) or os.path.getsize(file_path) == 0:
            print(f"*** SKIP *** missing/empty: {file_path}")
            continue
        data = load_json_with_continuations(file_path)
        if not data:
            continue
        for item in data:
            item["category"] = json_item.get("category", "")
            item["tag"] = json_item.get("tag", "")
            item["init"] = json_item.get("init", "")
            item["layer"] = json_item.get("layer", "")
            item["path"] = json_path
        all_data.extend(data)
    df = pd.DataFrame(all_data)
    df = df.explode("ft").reset_index(drop=True).dropna(subset=["ft"], ignore_index=True)
    df["stats"] = df["ft"].apply(lambda x: x["stats"])
    df["label"] = df.apply(lambda x: f"{x['init']}: {x['category']}: {x['layer']}: {x['tag']}", axis=1)
    df["seed"] = df["ft"].apply(lambda x: x["seed"])

    df_columns = df.columns.tolist()
    df = df.groupby(["label", "seed"]).agg({
        "stats": lists_extend,
        **{col: "first" for col in df_columns if col not in ["stats", "label", "seed"]}
    }).reset_index()

    df["layer_accuracies"] = df.apply(lambda x: get_layer_accuracies(x), axis=1)

    init_cols = []
    for ix in range(0, 12):
        for kind, layer_sub in [("attn", ix - 0.5), ("mlp", ix)]:
            col = f"delta_norm_ratio_{ix}{kind}"
            epochs_and_values = df.apply(
                lambda x: get_attribute_training_with_epochs(x, "delta_norm_ratio", layer_sub), axis=1
            )
            df[f"{col}_epochs"] = epochs_and_values.apply(lambda x: x[0])
            df[col] = epochs_and_values.apply(lambda x: x[1])

            init_col = f"{col}_init"
            init_epochs_and_values = df.apply(
                lambda x: get_attribute_training_with_epochs(x, "custom_delta_norm_ratio", layer_sub), axis=1
            )

            def _init_value(pair):
                epochs, values = pair
                for e, v in zip(epochs, values):
                    if e == -1:
                        return v
                return np.nan

            df[init_col] = init_epochs_and_values.apply(_init_value)
            init_cols.append(init_col)

    value_cols = [f"delta_norm_ratio_{ix}{kind}" for ix in range(12) for kind in ("attn", "mlp")]
    epoch_cols = [f"{c}_epochs" for c in value_cols]

    agg_dict = {
        "layer_accuracies": element_wise_mean,
        "category": "first",
        "init": "first",
        "tag": "first",
        "layer": "first",
        "seed": "count",
    }
    df_grouped = df.groupby(["label"]).agg(agg_dict).reset_index()
    df_grouped["final_accuracies"] = df.groupby(["label"]).agg(
        {"layer_accuracies": element_wise_mean}
    ).reset_index()["layer_accuracies"].apply(lambda x: x[-1] if len(x) > 0 else float("nan"))

    for col in value_cols:
        df_grouped[col] = df.groupby(["label"]).agg({col: element_wise_mean}).reset_index()[col]
    for ecol in epoch_cols:
        df_grouped[ecol] = df.groupby(["label"]).agg({ecol: "first"}).reset_index()[ecol]
    for icol in init_cols:
        df_grouped[icol] = df.groupby(["label"]).agg({icol: "mean"}).reset_index()[icol]

    return df_grouped


TASK3_FEATURES = ["init", "0-49", "e49", "50-99", "100-199", "200-299"]
TASK3_BUCKETS = {
    "0-49": (0, 50),
    "50-99": (50, 100),
    "100-199": (100, 200),
    "200-299": (200, 300),  # upper bound exclusive -> includes epoch 299
}


def _task3_align(epochs, values):
    epochs = np.asarray(epochs, dtype=float)
    values = np.asarray(values, dtype=float)
    n = min(len(epochs), len(values))
    return epochs[:n], values[:n]


def task3_bucket_median(epochs, values, low, high):
    epochs, values = _task3_align(epochs, values)
    mask = (epochs >= low) & (epochs < high)
    b_values = values[mask]
    return float(np.nanmedian(b_values)) if len(b_values) else np.nan


def task3_epoch_value(epochs, values, target_epoch):
    epochs, values = _task3_align(epochs, values)
    match = np.where(epochs == target_epoch)[0]
    return float(values[match[0]]) if len(match) else np.nan


def build_task3_feature_table(dft, group_name):
    """One row per model x component (layer, type): the 6 features + final_accuracy."""
    records = []
    for _, row in dft.iterrows():
        for ix in range(0, 12):
            for kind in ("attn", "mlp"):
                col = f"delta_norm_ratio_{ix}{kind}"
                epochs, values = row[f"{col}_epochs"], row[col]
                rec = {
                    "group": group_name,
                    "label": row["label"],
                    "component": f"L{ix}-{kind}",
                    "layer": ix,
                    "type": kind,
                    "final_accuracy": row["final_accuracies"],
                    "init": row[f"{col}_init"],
                    "e49": task3_epoch_value(epochs, values, 49),
                }
                for bname, (low, high) in TASK3_BUCKETS.items():
                    rec[bname] = task3_bucket_median(epochs, values, low, high)
                records.append(rec)
    return pd.DataFrame(records)


def task3_spearman(x, y):
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]
    if len(x) < 3 or np.std(x) == 0:
        return np.nan, np.nan
    res = spearmanr(x, y)
    return res.correlation, res.pvalue


def compute_task3_correlation_matrices(feat_df):
    """Spearman rho (and its p-value) of each of the 6 features against final_accuracy,
    per component (layer x attn/mlp) -- 24 rows x 6 columns."""
    components = [f"L{ix}-{kind}" for ix in range(12) for kind in ("attn", "mlp")]
    rho_mat = pd.DataFrame(index=components, columns=TASK3_FEATURES, dtype=float)
    rho_p_mat = pd.DataFrame(index=components, columns=TASK3_FEATURES, dtype=float)
    for comp in components:
        sub = feat_df[feat_df["component"] == comp]
        y = sub["final_accuracy"].to_numpy(dtype=float)
        for feat in TASK3_FEATURES:
            x = sub[feat].to_numpy(dtype=float)
            rho_mat.loc[comp, feat], rho_p_mat.loc[comp, feat] = task3_spearman(x, y)
    return rho_mat, rho_p_mat


SIGNIFICANCE_ALPHA = 0.05


def plot_task3_heatmap(mat, pmat, cbar_label, fname, output_dir=OUTPUT_DIR_TASK3):
    """Coefficient heatmap with non-significant cells (p >= SIGNIFICANCE_ALPHA, or
    undefined) visually faded out by a translucent white overlay, so significance reads
    at a glance instead of requiring a second pass over printed p-values."""
    # mat is 24 rows x 6 cols -- portrait, not the bundle's default landscape single-panel
    # size, so the figure height is set explicitly to fit all 24 row labels legibly while
    # the width and all font sizes still come from the tueplots bundle in the first cell.
    bundle_width, _ = plt.rcParams["figure.figsize"]
    fig, ax = plt.subplots(figsize=(bundle_width * 1.15, bundle_width * mat.shape[0] / mat.shape[1] * 0.5))
    sns.heatmap(
        mat, ax=ax, cmap="coolwarm", center=0, vmin=-1, vmax=1,
        annot=True, fmt=".2f", annot_kws={"fontsize": 13},
        linewidths=0.4, linecolor="white",
        cbar_kws={"label": cbar_label},
        yticklabels=1,
    )
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            p = pmat.iat[i, j]
            if pd.isna(p) or p >= SIGNIFICANCE_ALPHA:
                # zorder=5 puts this above the annotation text (zorder 3), not just above the
                # heatmap cell color (zorder 1) -- otherwise, for a cell whose coefficient is
                # already near 0 (pale under this diverging colormap), the overlay had nothing
                # visible to fade and the mask was effectively invisible.
                ax.add_patch(plt.Rectangle((j, i), 1, 1, facecolor="white", edgecolor="none", alpha=0.75, zorder=5))
    ax.set_xlabel(r"Feature (init $\rightarrow$ training $\rightarrow$ e49 $\rightarrow$ late training)", fontsize=18)
    ax.set_ylabel("Component (layer x attn/mlp)", fontsize=18)
    ax.tick_params(axis="x", rotation=45, labelsize=15)
    ax.tick_params(axis="y", rotation=0, labelsize=15)
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=15)
    cbar.ax.yaxis.label.set_fontsize(18)
    save_fig(fig, os.path.join(output_dir, fname))
    plt.show()
    return fig


def run_task3_for_group(group_name, json_map, parent_path_for_group, output_dir=OUTPUT_DIR_TASK3):
    """Task 3 for one group: builds the per-model x component feature table, computes the
    Spearman-rho / p-value matrices (24 components x 6 features), saves everything (feature
    CSV, matrix CSVs, the masked heatmap PNG+PDF, and a JSON with the matrix values so they
    can be reprocessed into a table or re-plotted later), and returns the intermediates."""
    print(f"Task 3 -- {group_name}: {len(json_map)} models")
    dft = process_jsons_training_with_init(json_map, parent_path_for_group)
    feat = build_task3_feature_table(dft, group_name)
    feat.to_csv(os.path.join(output_dir, f"{group_name}_task3_features.csv"), index=False)

    rho_mat, rho_p_mat = compute_task3_correlation_matrices(feat)
    rho_mat.to_csv(os.path.join(output_dir, f"{group_name}_task3_spearman.csv"))
    rho_p_mat.to_csv(os.path.join(output_dir, f"{group_name}_task3_spearman_pvalue.csv"))

    fig = plot_task3_heatmap(
        rho_mat, rho_p_mat, r"Spearman $\rho$", f"{group_name}_task3_spearman_heatmap", output_dir=output_dir
    )
    plt.close(fig)

    heatmap_payload = {
        "group": group_name,
        "components": list(rho_mat.index),
        "features": list(rho_mat.columns),
        "spearman_rho": rho_mat.where(pd.notnull(rho_mat), None).values.tolist(),
        "spearman_p": rho_p_mat.where(pd.notnull(rho_p_mat), None).values.tolist(),
        "significance_alpha": SIGNIFICANCE_ALPHA,
    }
    heatmap_json_path = os.path.join(output_dir, f"{group_name}_task3_spearman_heatmap.json")
    with open(heatmap_json_path, "w") as f:
        json.dump(heatmap_payload, f, indent=2)

    print(f"{group_name}: saved feature/matrix CSVs, heatmap PNG/PDF, and {heatmap_json_path}")
    return feat, rho_mat, rho_p_mat


def run_task3_pipeline(json_maps_by_subset, group_specs, output_dir=OUTPUT_DIR_TASK3):
    """Runs Task 3 for each spec in `group_specs`. Each spec is either a leaf/pooled-group
    name already present in `json_maps_by_subset` (post-flattening -- e.g. '2A_early', '0',
    '2_master'), or a (group_name, [leaf_names]) tuple to pool an arbitrary ad-hoc combination
    of leaves on the fly via `pool_leaf_subsets`.
    """
    flat = flatten_subsets(json_maps_by_subset)
    results = {}
    for spec in group_specs:
        if isinstance(spec, tuple):
            group_name, leaf_names = spec
            json_map = pool_leaf_subsets(json_maps_by_subset, leaf_names)
        else:
            group_name = spec
            if group_name not in flat:
                print(f"{group_name}: skipped (not found; available: {sorted(flat)})")
                continue
            json_map = flat[group_name]
        if not json_map:
            print(f"{group_name}: skipped (no data)")
            continue
        parent = PARENT_PATH_IMNET_BASE if _is_vitb_leaf(group_name) else PARENT_PATH_IMNET100_SMALL
        results[group_name] = run_task3_for_group(group_name, json_map, parent, output_dir)
    return results


# Smoke-test Task 3 on the smallest ViT-Small leaf first. To run the full sweep, extend this
# list, e.g.: run_task3_pipeline(json_maps_by_subset,
#     ["0", "2_master", "3_master", "5A", "5B", "5C", "5D", "5E", "5F", "6A", "6B"])
run_task3_pipeline(json_maps_by_subset, ["4A_early"])






## Task 4: Norm ratio through training, per subset (selected layers, colored by accuracy or label)

Recreates `report_plots.ipynb`'s final cell (one subplot per transformer layer x `attn`/`mlp`, `delta_norm_ratio` vs. epoch, one line + std band per model) but:

1. **Reuses `json_maps_by_subset`** instead of a hardcoded `json_map_main` -- pass any leaf/pooled-group name already registered there (`"2A_early"`, `"2_master"`, `"7_master"`, etc.), or a `(custom_name, [leaf_names])` tuple to pool an arbitrary ad-hoc combination of leaves on the fly via `pool_leaf_subsets`, exactly like Task 3's group specs.
2. **Restricts to selected layers** -- pass any subset of the 12 transformer layers (e.g. `[0, 1, 9, 10, 11]`), not all 12.
3. **Colors lines by `"label"`** (one fixed color per distinct model label, categorical palette, shared legend) **or by `"accuracy"`** (continuous colormap keyed to `final_accuracies`, one shared colorbar/scale across every subplot in the figure so the same color always means the same accuracy).

`resolve_json_map` does the subset/spec lookup; `plot_norm_ratio_through_training` draws the grid for an already-resolved `dft`; `run_norm_ratio_plot` is the one-call driver (mirrors `run_subset_pipeline`/`run_task3_pipeline`'s shape) that resolves the spec, builds `dft` via `process_jsons_training`, renders the plot, and saves it (`<name>_norm_ratio_L<layers>_<color_by>.png`/`.pdf`).

No default call is left running at the bottom of the code cell -- pick a subset/spec, layers, and `color_by` and call `run_norm_ratio_plot(...)` directly, the same way Task 2/3's subsets are chosen per run.


In [ ]:
def resolve_json_map(json_maps_by_subset, spec):
    """`spec` is either a leaf/pooled-group name string (looked up in json_maps_by_subset
    after flattening), or a `(custom_name, [leaf_names])` tuple to pool an ad-hoc combination
    of leaves on the fly via `pool_leaf_subsets`. Returns (name, json_map)."""
    if isinstance(spec, tuple):
        name, leaf_names = spec
        return name, pool_leaf_subsets(json_maps_by_subset, leaf_names)
    flat = flatten_subsets(json_maps_by_subset)
    if spec not in flat:
        raise KeyError(f"Unknown leaf/subset: {spec!r}. Available: {sorted(flat)}")
    return spec, flat[spec]


def plot_norm_ratio_through_training(dft, layers, color_by="label", title="", cmap_name="viridis", min_accuracy=None, vmin=83.5, vmax=86.5):
    """Norm ratio vs. epoch, one subplot per selected layer x attn/mlp (mirrors
    report_plots.ipynb's final cell), restricted to `layers`, with a mean line + std band
    per model. `color_by="label"` gives each distinct model label its own fixed color from a
    categorical palette (shared legend); `color_by="accuracy"` colors every line by its
    model's `final_accuracies` on one shared colormap scale across the whole figure (so the
    same color means the same accuracy in every subplot), with a shared colorbar instead.

    `min_accuracy`, if given, drops any model whose `final_accuracies` is below it (e.g. failed
    or degenerate runs stuck near chance level) before plotting. `vmin`/`vmax` set the
    color_by="accuracy" colorbar range (default tuned for ViT-Small's ~83-87% accuracies;
    pass e.g. vmin=77.5, vmax=80.5 for ViT-Base's ~77-80% range).
    """
    if color_by not in ("label", "accuracy"):
        raise ValueError(f"color_by must be 'label' or 'accuracy', got {color_by!r}")

    ncols = 2
    nrows = len(layers)
    # Figure kept deliberately small (see Task 2's grid plot) -- text reads bigger relative
    # to the figure once shrunk to fit a slide/A4 sheet than the same fonts on a huge figure.
    fig_width_in = 12.0
    fig_height_in = 3 * nrows
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(fig_width_in, fig_height_in), squeeze=False)

    if color_by == "label":
        labels = sorted(dft["label"].dropna().unique())
        palette = dict(zip(labels, sns.color_palette(n_colors=max(len(labels), 1))))
    else:
        cmap = plt.get_cmap(cmap_name)
        norm = plt.Normalize(vmin=vmin, vmax=vmax)

    for row_ix, layer in enumerate(layers):
        for col_ix, kind in enumerate(["attn", "mlp"]):
            ax = axes[row_ix, col_ix]
            col = f"delta_norm_ratio_{layer}{kind}"
            std_col = f"{col}_std"
            epoch_col = f"{col}_epochs"

            for _, row in dft.iterrows():
                if min_accuracy is not None and pd.notna(row["final_accuracies"]) and row["final_accuracies"] < min_accuracy:
                    continue
                epochs = np.asarray(row[epoch_col], dtype=float)
                values = np.asarray(row[col], dtype=float)
                std = np.asarray(row[std_col], dtype=float)
                if len(epochs) != len(values) or len(values) == 0:
                    continue

                if color_by == "label":
                    color = palette[row["label"]]
                else:
                    acc = row["final_accuracies"]
                    color = cmap(norm(acc)) if pd.notna(acc) else "grey"

                ax.plot(epochs, values, marker="o", markersize=1, linewidth=1, color=color)
                ax.fill_between(epochs, values - std, values + std, color=color, alpha=0.15)

            ax.set_title(f"Layer {layer} {kind}", fontsize=20)
            ax.set_xlabel("Epoch", fontsize=16)
            ax.set_ylabel("Norm ratio", fontsize=16)
            if layer==11 and kind=="attn":
                ax.set_ylim(0.0, 2.0)
            ax.set_xticks([0,50,100,150,200,250,299])
            ax.tick_params(labelsize=16)
            ax.grid(True, alpha=0.3)

    # Reserve vertical whitespace in *inches* (not figure-fraction) for the suptitle, and for
    # the legend (color_by="label" only) in the bottom-right corner -- same pattern as Task 2's
    # make_grid_plot, so the reserved band scales correctly regardless of how many layers/rows
    # this particular figure has.
    gap_in = 0.08
    suptitle_h_in = 0.4
    suptitle_y = 1 - gap_in / fig_height_in
    top_frac = max(0.5, suptitle_y - (suptitle_h_in + gap_in) / fig_height_in)

    if color_by == "label":
        ncol = min(len(labels), 3)
        legend_rows = -(-len(labels) // ncol)
        legend_h_in = 0.35 * legend_rows + 0.25
        bottom_frac = (legend_h_in + 2 * gap_in) / fig_height_in
        legend_y = gap_in / fig_height_in
        legend_handles = [
            mlines.Line2D([], [], color=palette[l], marker="o", linestyle="-", markersize=1, label=l)
            for l in labels
        ]
        fig.legend(
            handles=legend_handles, loc="lower right", bbox_to_anchor=(0.99, legend_y),
            bbox_transform=fig.transFigure, ncol=ncol, fontsize=8, title="Label", title_fontsize=9,
        )
        # fig.suptitle(title, fontsize=14, y=suptitle_y)
        fig.tight_layout(rect=(0, bottom_frac, 1, top_frac))
    else:
        # fig.suptitle(title, fontsize=14, y=suptitle_y)
        # fig.tight_layout(rect=(0, 0, 1, top_frac))
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=axes, fraction=0.05, pad=0.02)
        cbar.set_label("Final accuracy", fontsize=16)
        cbar.ax.tick_params(labelsize=16)

    return fig


def run_norm_ratio_plot(json_maps_by_subset, spec, layers, color_by="label", min_accuracy=None):
    """Task 4 driver: resolves `spec` (a subset/leaf name, or a (name, [leaf_names]) ad-hoc
    pool) into a json_map, builds its dft, renders the norm-ratio-through-training grid
    restricted to `layers`, saves it, and returns the figure."""
    global parent_path
    name, json_map = resolve_json_map(json_maps_by_subset, spec)
    if not json_map:
        print(f"{name}: skipped (no data)")
        return None

    parent_path = PARENT_PATH_IMNET_BASE if _is_vitb_leaf(name) else PARENT_PATH_IMNET100_SMALL

    print(f"=== {name}: norm ratio through training ({len(json_map)} entries, layers={layers}, color_by={color_by}) ===")
    dft = process_jsons_training(json_map)
    layer_tag = "-".join(str(l) for l in layers)
    # ViT-Base (Group 4) final accuracies sit around 77-80%, well below ViT-Small's ~83-87%
    # default colorbar range -- use Group 4's own range so its colors aren't all saturated.
    vmin, vmax = (77.5, 80.5) if _is_vitb_leaf(name) else (83.5, 86.5)
    fig = plot_norm_ratio_through_training(
        dft, layers=layers, color_by=color_by, title=f"{name} (layers {layer_tag})",
        min_accuracy=min_accuracy, vmin=vmin, vmax=vmax,
    )
    saved_paths = save_fig(fig, os.path.join(OUTPUT_DIR, f"{name}_norm_ratio_L{layer_tag}_{color_by}"))
    plt.close(fig)
    print(f"{name}: saved {', '.join(saved_paths.values())}")
    return fig


# Example (not run by default) -- pick a subset/spec, layers, and color_by, then call directly:
run_norm_ratio_plot(json_maps_by_subset, "1B", layers=[0, 1, 9, 10, 11], color_by="accuracy")
# run_norm_ratio_plot(json_maps_by_subset, ("my_custom", ["2A_early", "3B_late"]), layers=[0, 11], color_by="label")


## Task 5a: Layer-wise final accuracy per subset, with inset zoom + shading styling

Plots epoch-299 `layer_accuracies`, one line per model (`color_by="label"`/`"accuracy"`),
with three plot-level characteristics beyond a bare-axes line plot:

1. **Alternating white/grey attn/mlp shading bands** (`_add_layer_shading`) behind the 24
   x-positions.
2. **A zoomed inset** restricted to `inset_layers` (default `[10, 11]`), auto-placed in
   whichever of the 4 corners overlaps the least plotted data (`_pick_best_inset_corner`,
   an approximation of legend `loc="best"`, since `inset_axes` has no built-in equivalent),
   showing the same lines at that layer range's own auto-computed y-scale, plus a connector
   box + lines (`ax.indicate_inset_zoom`) linking it back to the zoomed region on the main
   axes.
3. **Task 6's x-axis tick style** -- one tick per layer (`0`-`11`), centered between its
   attn/mlp pair, instead of Task 5's 24 individual `"N attn"`/`"N mlp"` labels.

The inset always keeps all four spines, regardless of the notebook's global `NO_SPINES`
rcParam (set once in the first cell) -- without a full border the zoomed panel blends into
the main plot instead of reading as distinct.

`plot_layer_accuracies_v2` draws the figure for an already-built `dft`; `run_layer_accuracy_plot_v2`
is the one-call driver: resolves a subset/spec, builds `dft` via `process_jsons_training`,
renders, and saves.

**Three outputs per call.** Alongside the combined attn+mlp plot, `run_layer_accuracy_plot_v2`
also renders two single-kind companions via `plot_layer_accuracies_v2_split`: an attn-only
plot and an mlp-only plot, each with a 12-position x-axis (one point per layer, `0`-`11`)
instead of the combined plot's 24 attn/mlp positions. Since there's only one kind on the axis,
these drop the attn/mlp shading bands (moot with a single kind shown) but keep the zoomed
inset on `inset_layers`, now indexed directly by layer number. All three figures are saved as
PNG+PDF: `<name>_layer_accuracy_<color_by>_v2`, `..._v2_attn`, and `..._v2_mlp`.


In [ ]:
# Self-contained copies so this cell doesn't depend on any other Task cell having been run first.
from matplotlib.ticker import MaxNLocator
_ORDER_PRINT_EPOCH_GRADIENT = ["0-attn", "0-mlp", "1-attn", "1-mlp", "2-attn", "2-mlp", "3-attn", "3-mlp",
                               "4-attn", "4-mlp", "5-attn", "5-mlp", "6-attn", "6-mlp", "7-attn", "7-mlp",
                               "8-attn", "8-mlp", "9-attn", "9-mlp", "10-attn", "10-mlp", "11-attn", "11-mlp"]

_ATTN_SHADE_COLOR = "#FFFFFF"
_MLP_SHADE_COLOR = "#888888"

plt.rcParams.update(bundles.iclr2024(usetex=True, rel_width=0.49, nrows=1, ncols=1, family='serif'))
# Extract the calculated width, but manually boost the height
# w, h = plt.rcParams['figure.figsize']
# plt.rcParams['figure.figsize'] = (w, h * 1.4)


def _add_layer_shading(ax):
    """Light background bands behind each x-position: one color for attn columns,
    another for mlp columns, so attn/mlp stay visually distinguishable even though
    the tick labels only show the layer number."""
    for i, lbl in enumerate(_ORDER_PRINT_EPOCH_GRADIENT):
        color = _ATTN_SHADE_COLOR if lbl.endswith("attn") else _MLP_SHADE_COLOR
        ax.axvspan(i - 0.5, i + 0.5, color=color, alpha=0.1, zorder=0, lw=0)


def _pick_best_inset_corner(ax, xs_all, ys_all, width=0.42, height=0.40, margin=0.05):
    """Picks whichever of the 4 corners (as an [x, y, width, height] axes-fraction bbox)
    contains the fewest of the already-plotted (xs_all, ys_all) points -- approximates
    legend loc="best" for an inset_axes, which has no built-in equivalent. Ties (including
    the empty-data case) prefer top-left, then top-right, bottom-left, bottom-right, in that
    scan order."""
    candidates = [
        ("top-left", margin, 1 - margin - height),
        ("top-right", 1 - margin - width, 1 - margin - height),
        ("bottom-left", margin, margin),
        ("bottom-right", 1 - margin - width, margin),
    ]
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    xs_all = np.asarray(xs_all, dtype=float)
    ys_all = np.asarray(ys_all, dtype=float)

    best_bbox, best_count = None, None
    for _name, fx, fy in candidates:
        cx0, cx1 = x0 + fx * (x1 - x0), x0 + (fx + width) * (x1 - x0)
        cy0, cy1 = y0 + fy * (y1 - y0), y0 + (fy + height) * (y1 - y0)
        mask = (xs_all >= cx0) & (xs_all <= cx1) & (ys_all >= cy0) & (ys_all <= cy1)
        count = int(mask.sum())
        if best_count is None or count < best_count:
            best_bbox, best_count = (fx, fy, width, height), count
    return best_bbox


def plot_layer_accuracies_v2(dft, color_by="label", cmap_name="viridis", title="", min_accuracy=None, vmin=83.5, vmax=86.5, inset_layers=(10, 11), legend=True):
    """Task 5a: same data as Task 5's plot_layer_accuracies (epoch-299 layer-wise accuracy,
    one line per model) with Task 6's plot-level formatting -- alternating white/grey
    attn/mlp shading bands, the layer-number x-axis tick style (one tick per layer, centered
    between its attn/mlp pair), and an inset zoomed into `inset_layers`, auto-placed in
    whichever corner overlaps the least plotted data (like legend loc="best"), with a
    connector box + lines (matplotlib's indicate_inset_zoom) linking it back to the zoomed
    region on the main axes.
    """
    if color_by not in ("label", "accuracy"):
        raise ValueError(f"color_by must be 'label' or 'accuracy', got {color_by!r}")

    n_positions = len(_ORDER_PRINT_EPOCH_GRADIENT)
    x = np.arange(n_positions)

    if color_by == "label":
        fig, ax = plt.subplots()
    else:
        fig, ax = plt.subplots()

    _add_layer_shading(ax)

    if color_by == "label":
        labels = sorted(dft["label"].dropna().unique())
        palette = dict(zip(labels, sns.color_palette(n_colors=max(len(labels), 1))))
    else:
        cmap = plt.get_cmap(cmap_name)
        norm = plt.Normalize(vmin=vmin, vmax=vmax)

    def _draw(target_ax):
        for _, row in dft.iterrows():
            if min_accuracy is not None and pd.notna(row["final_accuracies"]) and row["final_accuracies"] < min_accuracy:
                continue
            y = np.asarray(row["layer_accuracies"], dtype=float)
            if len(y) == 0:
                continue
            if len(y) != n_positions:
                print(f"*** WARNING ***: expected {n_positions} layer accuracies, got {len(y)} "
                      f"for label={row['label']!r}; skipping.")
                continue
            if color_by == "label":
                color = palette[row["label"]]
            else:
                acc = row["final_accuracies"]
                color = cmap(norm(acc)) if pd.notna(acc) else "grey"
            target_ax.plot(x, y, marker="o", markersize=1, linewidth=1, color=color)

    _draw(ax)

    tick_positions = [i + 0.5 for i in range(0, n_positions, 2)]
    tick_labels = [str(i) for i in range(n_positions // 2)]
    ax.set_xlim(-0.5, n_positions - 0.5)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=0)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Logit lens accuracy")
    ax.grid(True, alpha=0.3)

    inset_positions = [i for i, lbl in enumerate(_ORDER_PRINT_EPOCH_GRADIENT) if int(lbl.split("-")[0]) in inset_layers]
    if inset_positions:
        xs_all, ys_all = [], []
        for _, row in dft.iterrows():
            y = row["layer_accuracies"]
            if not isinstance(y, list) or len(y) != n_positions:
                continue
            xs_all.extend(x.tolist())
            ys_all.extend(y)

        fx, fy, iw, ih = _pick_best_inset_corner(ax, xs_all, ys_all)
        axins = ax.inset_axes([fx, fy, iw, ih])
        _add_layer_shading(axins)
        _draw(axins)

        lo, hi = min(inset_positions), max(inset_positions)
        axins.set_xlim(lo - 0.5, hi + 0.5)

        vals = []
        for _, row in dft.iterrows():
            y = row["layer_accuracies"]
            if not isinstance(y, list) or len(y) != n_positions:
                continue
            for p in inset_positions:
                v = y[p]
                if v is not None and not (isinstance(v, float) and np.isnan(v)):
                    vals.append(v)
        if vals:
            lo_v, hi_v = min(vals), max(vals)
            pad = max(hi_v - lo_v, 1.0) * 0.1
            axins.set_ylim(max(0, lo_v - pad), min(100, hi_v + pad))

        even_start = lo - (lo % 2)
        inset_tick_positions = [i + 0.5 for i in range(even_start, hi + 1, 2)]
        inset_tick_labels = [str(i // 2) for i in range(even_start, hi + 1, 2)]
        axins.set_xticks(inset_tick_positions)
        axins.set_xticklabels(inset_tick_labels)
        axins.tick_params(axis='y')
        axins.set_xlabel("")
        axins.set_ylabel("")
        axins.grid(alpha=0.3)
        # When the inset sits in a left-hand corner, its default left-side y-tick labels land
        # right on top of the main axes' own left-side y-axis labels -- flip them to the
        # inset's right edge (pointing inward, away from the main axis) instead.
        if fx <= 0.5:
            axins.yaxis.tick_right()
        # Unlike the main axes (which follow the notebook's global NO_SPINES rcParam), the
        # inset always keeps all four spines -- without them its border blends into the main
        # plot instead of reading as a distinct zoomed-in panel.
        for spine in axins.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.6)
        # Connector box + lines linking the inset back to the region it zooms into on the
        # main axes (only after axins' xlim/ylim are final, since this reads them).
        ax.indicate_inset_zoom(axins, edgecolor="black", linewidth=0.8, alpha=0.6)
    else:
        print(f"No layer positions matched inset_layers={inset_layers}; skipping inset.")

    if color_by == "label" and legend:
        # Legend placed above the axes, same style as Task 7's color legend -- ax.legend()
        # with loc="lower center" anchored just above the plot (bbox_to_anchor y=1), unframed.
        ncol = min(len(labels), 3)
        legend_handles = [
            mlines.Line2D([], [], color=palette[l], marker="o", linestyle="-", markersize=1, label=l.replace(":", "").strip())
            for l in labels
        ]
        ax.legend(
            handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, 1),
            ncol=ncol, frameon=False,
        )
    elif color_by == "accuracy":
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.05, pad=0.02)
        cbar.set_label("Final accuracy")
        cbar.ax.yaxis.set_major_locator(MaxNLocator(integer=True))

    return fig


def plot_layer_accuracies_v2_split(dft, kind, color_by="label", cmap_name="viridis", title="", min_accuracy=None, vmin=83.5, vmax=86.5, inset_layers=(10, 11), legend=True):
    """Single-kind ("attn"-only or "mlp"-only) companion to plot_layer_accuracies_v2.

    Restricts `layer_accuracies` to just the 12 positions of one kind (attn or mlp), so the
    x-axis is one point per layer (0-11) instead of the combined plot's 24 attn/mlp positions.
    Since there is only one kind on the axis, the attn/mlp shading bands are dropped (they
    exist to distinguish attn columns from mlp columns, which is moot here); the zoomed inset
    on `inset_layers` is kept, now indexed directly by layer number since x already is layer
    number. Legend/colorbar handling is identical to plot_layer_accuracies_v2.
    """
    if kind not in ("attn", "mlp"):
        raise ValueError(f"kind must be 'attn' or 'mlp', got {kind!r}")
    if color_by not in ("label", "accuracy"):
        raise ValueError(f"color_by must be 'label' or 'accuracy', got {color_by!r}")

    n_layers = 12
    offset = 0 if kind == "attn" else 1
    x = np.arange(n_layers)

    if color_by == "label":
        fig, ax = plt.subplots()
    else:
        fig, ax = plt.subplots()

    if color_by == "label":
        labels = sorted(dft["label"].dropna().unique())
        palette = dict(zip(labels, sns.color_palette(n_colors=max(len(labels), 1))))
    else:
        cmap = plt.get_cmap(cmap_name)
        norm = plt.Normalize(vmin=vmin, vmax=vmax)

    def _draw(target_ax):
        for _, row in dft.iterrows():
            if min_accuracy is not None and pd.notna(row["final_accuracies"]) and row["final_accuracies"] < min_accuracy:
                continue
            y_full = row["layer_accuracies"]
            if not isinstance(y_full, list) or len(y_full) != len(_ORDER_PRINT_EPOCH_GRADIENT):
                print(f"*** WARNING ***: expected {len(_ORDER_PRINT_EPOCH_GRADIENT)} layer accuracies, got "
                      f"{len(y_full) if isinstance(y_full, list) else y_full!r} for label={row['label']!r}; skipping.")
                continue
            y = np.asarray([y_full[2 * i + offset] for i in range(n_layers)], dtype=float)
            if color_by == "label":
                color = palette[row["label"]]
            else:
                acc = row["final_accuracies"]
                color = cmap(norm(acc)) if pd.notna(acc) else "grey"
            target_ax.plot(x, y, marker="o", markersize=1, linewidth=1, color=color)

    _draw(ax)

    ax.set_xlim(-0.5, n_layers - 0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([str(i) for i in range(n_layers)], rotation=0)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Logit lens accuracy")
    ax.grid(True, alpha=0.3)

    inset_layer_positions = [i for i in range(n_layers) if i in inset_layers]
    if inset_layer_positions:
        xs_all, ys_all = [], []
        for _, row in dft.iterrows():
            y_full = row["layer_accuracies"]
            if not isinstance(y_full, list) or len(y_full) != len(_ORDER_PRINT_EPOCH_GRADIENT):
                continue
            y = [y_full[2 * i + offset] for i in range(n_layers)]
            xs_all.extend(x.tolist())
            ys_all.extend(y)

        fx, fy, iw, ih = _pick_best_inset_corner(ax, xs_all, ys_all)
        axins = ax.inset_axes([fx, fy, iw, ih])
        _draw(axins)

        lo, hi = min(inset_layer_positions), max(inset_layer_positions)
        axins.set_xlim(lo - 0.5, hi + 0.5)

        vals = []
        for _, row in dft.iterrows():
            y_full = row["layer_accuracies"]
            if not isinstance(y_full, list) or len(y_full) != len(_ORDER_PRINT_EPOCH_GRADIENT):
                continue
            for p in inset_layer_positions:
                v = y_full[2 * p + offset]
                if v is not None and not (isinstance(v, float) and np.isnan(v)):
                    vals.append(v)
        if vals:
            lo_v, hi_v = min(vals), max(vals)
            pad = max(hi_v - lo_v, 1.0) * 0.1
            axins.set_ylim(max(0, lo_v - pad), min(100, hi_v + pad))

        axins.set_xticks(list(range(lo, hi + 1)))
        axins.set_xticklabels([str(i) for i in range(lo, hi + 1)])
        axins.tick_params(axis='y', labelsize=6)
        axins.set_xlabel("")
        axins.set_ylabel("")
        axins.grid(alpha=0.3)
        # Same left-corner y-tick flip as plot_layer_accuracies_v2, for the same reason.
        if fx <= 0.5:
            axins.yaxis.tick_right()
        # Inset always keeps all four spines regardless of the notebook's global NO_SPINES
        # rcParam, same as plot_layer_accuracies_v2's inset.
        for spine in axins.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.6)
        ax.indicate_inset_zoom(axins, edgecolor="black", linewidth=0.8, alpha=0.6)
    else:
        print(f"No layers matched inset_layers={inset_layers}; skipping inset.")

    if color_by == "label" and legend:
        # Same Task-7-style legend as plot_layer_accuracies_v2: ax.legend() anchored just
        # above the axes, unframed.
        ncol = min(len(labels), 3)
        legend_handles = [
            mlines.Line2D([], [], color=palette[l], marker="o", linestyle="-", markersize=1, label=l)
            for l in labels
        ]
        ax.legend(
            handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, 1),
            ncol=ncol, frameon=False,
        )
    elif color_by == "accuracy":
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.05, pad=0.02)
        cbar.set_label("Final accuracy")
        cbar.ax.yaxis.set_major_locator(MaxNLocator(integer=True))

    return fig


def run_layer_accuracy_plot_v2(json_maps_by_subset, spec, color_by="label", min_accuracy=None,
                                inset_layers=(10, 11), inset_layers_attn=None, inset_layers_mlp=None,
                                legend=True):
    """Task 5a driver: same shape as Task 5's run_layer_accuracy_plot, but renders via
    plot_layer_accuracies_v2 (Task 6's plot-level formatting) instead of plot_layer_accuracies.

    Always renders and saves three figures: the combined 24-position plot
    (`<name>_layer_accuracy_<color_by>_v2`), and single-kind companions restricted to just the
    attn positions (`..._v2_attn`) or just the mlp positions (`..._v2_mlp`), via
    plot_layer_accuracies_v2_split.

    `inset_layers` sets the combined plot's inset range and is also the default for the attn/mlp
    companions; pass `inset_layers_attn`/`inset_layers_mlp` to override just one of those two
    without changing the combined plot or the other companion.
    """
    global parent_path
    name, json_map = resolve_json_map(json_maps_by_subset, spec)
    if not json_map:
        print(f"{name}: skipped (no data)")
        return None

    parent_path = PARENT_PATH_IMNET_BASE if _is_vitb_leaf(name) else PARENT_PATH_IMNET100_SMALL

    print(f"=== {name}: layer-wise final accuracy v2 ({len(json_map)} entries, color_by={color_by}) ===")
    dft = process_jsons_training(json_map)
    # vmin, vmax = (78, 80.5) if name.startswith("4") else (83.5, 86.5)
    vmin, vmax = dft["final_accuracies"].min(), dft["final_accuracies"].max()

    fig = plot_layer_accuracies_v2(
        dft, color_by=color_by, title=f"{name}: layer-wise accuracy", min_accuracy=min_accuracy,
        vmin=vmin, vmax=vmax, inset_layers=inset_layers, legend=legend,
    )
    saved_paths = save_fig(fig, os.path.join(OUTPUT_DIR, f"{name}_layer_accuracy_{color_by}_v2"))
    plt.close(fig)
    print(f"{name}: saved {', '.join(saved_paths.values())}")

    inset_layers_by_kind = {
        "attn": inset_layers_attn if inset_layers_attn is not None else inset_layers,
        "mlp": inset_layers_mlp if inset_layers_mlp is not None else inset_layers,
    }
    for kind in ("attn", "mlp"):
        fig_kind = plot_layer_accuracies_v2_split(
            dft, kind, color_by=color_by, title=f"{name}: layer-wise accuracy ({kind})",
            min_accuracy=min_accuracy, vmin=vmin, vmax=vmax, inset_layers=inset_layers_by_kind[kind],
            legend=legend,
        )
        saved_paths_kind = save_fig(fig_kind, os.path.join(OUTPUT_DIR, f"{name}_layer_accuracy_{color_by}_v2_{kind}"))
        plt.close(fig_kind)
        print(f"{name}: saved {', '.join(saved_paths_kind.values())}")

    return fig


# Example (not run by default) -- pick a subset/spec, color_by, and inset_layers, then call directly:
for s in ["4A_early", "4A_late", "7A"]:
    run_layer_accuracy_plot_v2(json_maps_by_subset, s, color_by="accuracy", inset_layers=[11], inset_layers_attn=[11], inset_layers_mlp=[10.5, 11], legend=False)
# run_layer_accuracy_plot_v2(json_maps_by_subset, "4A_late", color_by="label", inset_layers=[11], inset_layers_attn=[11], inset_layers_mlp=[10.5, 11], legend=True)
# run_layer_accuracy_plot_v2(json_maps_by_subset, "1B", color_by="label", inset_layers=[0, 1])
run_layer_accuracy_plot_v2(json_maps_by_subset, ("2A", ["2A_early", "2A_late", "1A"]), color_by="accuracy", inset_layers=[11], inset_layers_attn=[11], inset_layers_mlp=[10.5, 11])



## Task 7: layer-wise accuracy over epochs

For a configurable set of raw `layer_sub` values (default `[4, 7, 11]`), plots probe accuracy (`acc`) vs. epoch as one line per (json, layer) pair:

- **Color** encodes the json/run (`json_map_task7` entry).
- **Linestyle** encodes the layer (`LAYERS_TO_PLOT`).
- Multiple json entries are overlaid on the same axes.
- Accuracy is averaged across seeds at each epoch for a given (json, layer).
- Epoch `-1` (the pre-training snapshot) is dropped; the x-axis starts at epoch 0.

Configure `json_map_task7` below using the same `init` / `category` / `layer` / `tag` / `path` keys as the other `json_map_*` lists in this notebook — `process_jsons_training` derives `row["label"]` from those (`f"{init}: {category}: {layer}: {tag}"`), the same convention used everywhere else, rather than a standalone `"label"` key.


In [ ]:
# parent_path_task7 is set explicitly instead of relying on whatever an earlier cell in this
# notebook last left the global `parent_path` pointing at -- process_jsons_layer_acc_over_epochs
# below takes parent_path as an explicit argument instead.

LAYERS_TO_PLOT = [4, 7, 11]  # configurable raw layer_sub values

parent_path_task7 = PARENT_PATH_IMNET_BASE
json_map_task7 = [
    # {
    #     "init": "k-Dyck - D4",
    #     "category": "PR",
    #     "layer": "L0-8",
    #     "path": "s6316138.json",
    # },
    # {
    #     "init": "Random-init",
    #     "category": "",
    #     "layer": "",
    #     "path": "6257433",
    # },
    {
        'category': '',
        'layer': '',
        'path': 'accuracy_IMNET_BASE_29384839.json',
        'init': 'Random init',
    },
    {
        'category': '',
        'layer': '',
        'path': 'accuracy_IMNET_BASE_29377576.json',
        'init': 'k-Dyck - D4',
    },
    {
        "init": "k-Dyck Shuffled - D98",
        "category": "",
        "path": "ftb4_d98_accuracy_IMNET_BASE_29547831.json"
    },
]
# json_map_task7 = [
#     {
#         'category': 'Random init w/ PW properties',
#         'layer': '',
#         'path': 'accuracy_IMNET_BASE_29737095_s0.json',
#         'init': 'k-Dyck - D4',
#     },
#     {
#         'category': 'Random init w/ PW properties',
#         'layer': '',
#         'path': 'accuracy_IMNET_BASE_29736861_s0.json',
#         'init': 'k-Dyck Shuffled - D98',
#     },
# ]


# parent_path_task7 = PARENT_PATH_IMNET100_SMALL
# json_map_task7 = [
#     {'init': 'Random-init', 'path': 's6257433.json'},
#     {"init": "k-Dyck - D4", "category": "PR", "layer": "L0-11", "path": "s6848384.json"},
#     {"init": "k-Dyck - D98", "category": "PR", "layer": "L0-11", "path": "s6848383.json"},  # new (was s4922828.json)
#     {"init": "k-Dyck Truncated - D4", "category": "PR", "layer": "L0-11", "path": "s6848389.json"},  # new (was s4922830.json)
#     {"init": "k-Dyck Truncated - D98", "category": "PR", "layer": "L0-11", "path": "s6848392.json"},
#     {"init": "k-Dyck Shuffled - D4", "category": "PR", "layer": "L0-11", "path": "s6848393.json"},  # new (was s4922832.json)
#     {"init": "k-Dyck Shuffled - D98", "category": "PR", "layer": "L0-11", "path": "s6848396.json"},
#     {"init": "k-Dyck Truncated Shuffled - D4", "category": "PR", "layer": "L0-11", "path": "s6848399.json"},
#     {"init": "k-Dyck Truncated Shuffled - D98", "category": "PR", "layer": "L0-11", "path": "s6848400.json"}
#   ]

def process_jsons_layer_acc_over_epochs(json_map, parent_path_for_group, layers):
    """Task 7 driver: loads each json's per-seed rows the same way process_jsons_training
    does, but keeps the per-epoch 'acc' series (via get_attribute_training_with_epochs) for
    each requested raw layer_sub value through to the grouped-by-label output, averaged
    across seeds with the existing element_wise_mean helper.

    process_jsons_training can't be reused directly here: its own final groupby only carries
    forward layer_accuracies/category/init/tag/layer/path/seed plus the delta_norm_ratio
    columns it explicitly loops over -- it drops the per-seed 'stats' column entirely, so
    there's no way to pull an arbitrary column like 'acc' back out of its returned dataframe.
    """
    all_data = []
    for json_item in json_map:
        json_path = json_item["path"]
        print(f"Processing {json_path}...")
        file_path = os.path.join(parent_path_for_group, json_path)
        if not os.path.exists(file_path) or os.path.getsize(file_path) == 0:
            print(f"*** SKIP *** missing/empty: {file_path}")
            continue
        data = load_json_with_continuations(file_path)
        if not data:
            continue
        for item in data:
            item["category"] = json_item.get("category", "")
            item["tag"] = json_item.get("tag", "")
            item["init"] = json_item.get("init", "")
            item["layer"] = json_item.get("layer", "")
            item["path"] = json_path
        all_data.extend(data)
    if not all_data:
        raise ValueError(f"*** ERROR ***: No data was loaded for any file in json_map: {[j.get('path') for j in json_map]}")
    df = pd.DataFrame(all_data)

    df = df.explode("ft").reset_index(drop=True).dropna(subset=["ft"], ignore_index=True)
    df["stats"] = df["ft"].apply(lambda x: x["stats"])
    df["label"] = df.apply(lambda x: f"{x['init']}: {x['category']}: {x['layer']}: {x['tag']}", axis=1)
    df["seed"] = df["ft"].apply(lambda x: x["seed"])

    df_columns = df.columns.tolist()
    df = df.groupby(["label", "seed"]).agg({
        "stats": lists_extend,
        **{col: "first" for col in df_columns if col not in ["stats", "label", "seed"]}
    }).reset_index()

    acc_cols = []
    epoch_cols = []
    for layer in layers:
        col = f"acc_{layer}"
        epochs_and_values = df.apply(lambda x: get_attribute_training_with_epochs(x, "acc", layer), axis=1)
        df[f"{col}_epochs"] = epochs_and_values.apply(lambda x: x[0])
        df[col] = epochs_and_values.apply(lambda x: x[1])
        acc_cols.append(col)
        epoch_cols.append(f"{col}_epochs")

    df_grouped = df.groupby(["label"]).agg({
        "category": "first", "init": "first", "tag": "first", "layer": "first", "path": "first", "seed": "count",
    }).reset_index()

    for col in acc_cols:
        df_grouped[col] = df.groupby(["label"]).agg({col: element_wise_mean}).reset_index()[col]
    # epochs should line up across seeds for the same label; just take the first seed's epoch list
    for ecol in epoch_cols:
        df_grouped[ecol] = df.groupby(["label"]).agg({ecol: "first"}).reset_index()[ecol]

    if any(df_grouped["seed"] != 3):
        print("Warning: Some groups do not have 3 seeds. Here are those rows:")
        print(df_grouped[df_grouped["seed"] != 3])

    return df_grouped


dft_task7 = process_jsons_layer_acc_over_epochs(json_map_task7, parent_path_task7, LAYERS_TO_PLOT)
dft_task7.columns


In [ ]:
records = []
for _, row in dft_task7.iterrows():
    label = row["label"]
    for layer in LAYERS_TO_PLOT:
        epochs = row[f"acc_{layer}_epochs"]
        values = row[f"acc_{layer}"]
        for epoch, value in zip(epochs, values):
            if epoch < 0:
                continue
            records.append({
                "json_label": label.replace(":", "").strip(),
                "layer": layer,
                "epoch": epoch,
                "acc": value,
            })

if not records:
    raise ValueError(
        "*** ERROR ***: No accuracy records found for json_map_task7 / LAYERS_TO_PLOT. "
        "Check that json_map_task7 is populated and that the chosen layers exist as layer_sub values."
    )

# Already averaged across seeds (one row per label/model in dft_task7 via element_wise_mean),
# so no further groupby is needed here -- unlike a naive per-seed long dataframe would.
df_task7_avg = pd.DataFrame(records)
df_task7_avg


In [ ]:
from matplotlib.lines import Line2D
plt.rcParams.update(bundles.iclr2024(usetex=True, rel_width=0.49, nrows=1, ncols=1, family='serif'))
# Extract the calculated width, but manually boost the height
# w, h = plt.rcParams['figure.figsize']
# plt.rcParams['figure.figsize'] = (w, h * 1)
# plt.rcParams['figure.constrained_layout.use'] = False

json_labels = df_task7_avg["json_label"].unique().tolist()
# Default seaborn categorical palette, same as every other per-label coloring in this
# notebook (e.g. Task 4's plot_norm_ratio_through_training), rather than a different named
# colormap picked just for this plot.
color_palette = sns.color_palette(n_colors=max(len(json_labels), 1))
color_map = dict(zip(json_labels, color_palette))

linestyle_cycle = [":", "--", "-"]
linestyle_map = {
    layer: linestyle_cycle[i % len(linestyle_cycle)]
    for i, layer in enumerate(LAYERS_TO_PLOT)
}

# layout=None disables tueplots' constrained-layout engine (set globally by
# bundles.iclr2024() in the first cell) just for this figure -- constrained layout silently
# ignores fig.subplots_adjust() below, which is what left the two legend rows overlapping
# # each other and the axes.
fig, ax = plt.subplots()
for (json_label, layer), sub_df in df_task7_avg.groupby(["json_label", "layer"]):
    sub_df = sub_df.sort_values("epoch")
    ax.plot(
        sub_df["epoch"],
        sub_df["acc"],
        color=color_map[json_label],
        linestyle=linestyle_map[layer],
    )

ax.set_xlabel("Epoch")
ax.set_ylabel("Logit lens accuracy")
# ax.set_title("Layer-wise accuracy over epochs")
ax.set_xticks([0, 50, 100, 150, 200, 250, 299])
ax.grid(True, alpha=0.3)

color_handles = [
    Line2D([0], [0], color=color_map[lbl], linestyle="-", label=lbl)
    for lbl in json_labels
]
style_handles = [
    Line2D([0], [0], color="black", linestyle=linestyle_map[layer], label=f"{layer}")
    for layer in LAYERS_TO_PLOT
]

style_legend = ax.legend(
    handles=style_handles, 
    loc="lower center", 
    bbox_to_anchor=(0.5, 1), # Anchors the bottom of the text 2% above the plot
    ncol=len(LAYERS_TO_PLOT), 
    frameon=False
)


layer_tag = "-".join(str(l) for l in LAYERS_TO_PLOT)
save_basename = os.path.join(OUTPUT_DIR, f"VITB_baseline_layer_acc_over_epochs_L{layer_tag}")
saved_paths = {}
for fmt in ("png", "pdf"):
    saved_paths.update(save_fig(fig, save_basename, formats=(fmt,)))
print(f"Saved {', '.join(saved_paths.values())}")

plt.show()
